# Flow Matching tutorial: training ODE generative models like Diffusion models

## Introduction

**Flow Matching was introduced in three different ICLR 2023 papers and has drawn a lot of attention in the machine learning community recently. We would like to highlight all of them here: Flow Matching [(Lipman et al.)](https://arxiv.org/abs/2210.02747), Stochastic Interpolants [(Albergo et al.)](https://arxiv.org/abs/2209.15571) and Rectified Flow [(Liu et al.)](https://arxiv.org/abs/2209.03003).**

Flow Matching, a recently introduced generative model, leverages an ordinary differential equation (ODE) to mold a base density into the desired data distribution. In contrast, diffusion is based on a stochastic differential equation (SDE). This notebook illustrates the training of Flow Matching methods, highlighting key components. In this notebook, we present two Flow Matching models built upon the original formulation: Independent Conditional Flow Matching (I-CFM) and Optimal Transport Conditional Flow Matching (OT-CFM).

In our notation, $\alpha$ represents the noise distribution, typically a Gaussian, while $\beta$ denotes the distribution corresponding to real data.

Note from the authors: this is a beta tutorial! Do not hesitate to suggest improvements through the opened issue https://github.com/atong01/conditional-flow-matching/issues/88

## ODE: transforming a data distribution into another one

Let's delve into some foundational concepts.  We consider a smooth time-varying vector field $u : [0, 1] \times \mathbb{R}^d \to \mathbb{R}^d$ that defines an ordinary differential equation:
$$
dx = u_t(x)\,dt.
$$

We denote by $\phi_t(x)$ the solution of the ODE with initial condition $\phi_0(x)=x$. In essence, $\phi_t(x)$ represents the trajectory of a point $x$ transported along the vector field $u$ from time $0$ to $t$.

Assuming the knowledge of the distributions $p_0$ and $p_1$, a transformative process between them can be initiated. The integration map $\phi_t$ induces a pushforward measure $p_t:=[\phi_t]_\#(p_0)$ between $p_0$ and $p_1$. This measure characterizes the density of points $x\sim p_0$ transported by $u$ from time $0$ to time $t$. The time-varying density $p_t$, viewed as a function $p:[0,1]\times\mathbb{R}^d\to\mathbb{R}$, is characterized by the initial conditions $\phi_0(x)=x$ and the well-known $\textbf{continuity equation}$:
\begin{equation}
\frac{\partial p}{\partial t}=-\nabla\cdot(p_t u_t)
\end{equation}

## Regressing the vector field: an intractable loss

Suppose we have knowledge of the probability path $p_t(x)$ and the associated vector field $u_t(x)$ that generates it, with the added convenience that $p_t(x)$ is readily samplable.

If $v_\theta(\cdot,\cdot):[0,1]\times\mathbb{R}^d\to\mathbb{R}^d$ is a time-dependent vector field parametrized as a neural network (with weights $\theta$), we aim to regress $v_\theta$ to $u$ using the $\textbf{flow matching (FM)}$ objective:

$$
\mathcal{L}_{\text{FM}}(\theta) := \mathbb{E}_{t\sim [0,1],x\sim p_t(x)} \| v_\theta(t, x) - u_t(x) \|^2.
$$

This objective allows us to align $v_\theta$ with $u$ by minimizing the expected squared norm difference across various time points $t$ and sampled points $x$ from $p_t(x)$. Then we can generate data by using $v_\theta$ in place of $u$ in the above ODE. However, this objective becomes intractable for general source and target distributions as $p_t(x)$ and $u_t$ are unknown general functions.

## Conditional Flow Matching

The primary limitation of the above approach is that it is untractable as we do not know $u_t$ and $p_t$. To render the objective feasible, we opt for a more manageable strategy by specifying the form of the probability path and vector field.

Here, we assume that the probability path $p_t(x)$ takes the form of a mixture involving **conditional probability paths $p_t(x|z)$**, where $z$ serves as a conditioning variable:

\begin{equation}\label{eq:ppath}
p_t(x) := \int p_t(x | z) q(z)\, dz.
\end{equation}
$q(z)$ represents a distribution over the conditioning variable. Furthermore, we want $p_1(x | z)$ to be a distribution concentrated around $z$.  A natural design choise for the conditional probability path $p_t(x|z)$ is to adopt Gaussian distributions, expressed as $p_t(x | z) = \mathcal{N}(\mu_t(z), \sigma_t^2)$. In our paper, we choose to condition over a tuple of source and target samples $z = (x_0, x_1) \sim \alpha \otimes \beta$. We define $\mu_t$ as the linear interpolation between $x_0$ and $x_1$ (with respect to time, *i.e.,* $\mu_t(x_0, x_1) = t x_1 + (1 - t) x_0$), and set the standard deviation $\sigma_t = \sigma>0$ as a constant float. The motivation behind this specific $\mu_t$ lies in the optimal transport theory that we will detail later. It's worth noting that alternate selections are possible as highlithed in (Lipman et al, 2023).

Unfortunately, direct sampling from the unconditional probability path $p_t(x)$ is hindered by the integral in its formulation. Instead, we will sample from the conditional probability path $p_t(x|z)$ and show later that we can build a conditional Flow Matching loss which shares a lot of benefits with its unconditional counter part.

In [ ]:
def sample_conditional_pt(x0, x1, t, sigma):
    """
    Draw a sample from the probability path N(t * x1 + (1 - t) * x0, sigma), see (Eq.14) [1].

    Parameters
    ----------
    x0 : Tensor, shape (bs, *dim)
        represents the source minibatch
    x1 : Tensor, shape (bs, *dim)
        represents the target minibatch
    t : FloatTensor, shape (bs)

    Returns
    -------
    xt : Tensor, shape (bs, *dim)

    References
    ----------
    [1] Improving and Generalizing Flow-Based Generative Models with minibatch optimal transport, Preprint, Tong et al.
    """
    t = t.reshape(-1, *([1] * (x0.dim() - 1)))
    mu_t = t * x1 + (1 - t) * x0
    epsilon = torch.randn_like(x0)
    return mu_t + sigma * epsilon

## Building an admissible vector field

We proceed with the assumption that the conditional probability paths $p_t(x|z)$, used to define the unconditional probability path, are generated by some conditional vector field $u_t(x|z)$.

The next challenge is to define an unconditional vector field $u_t(x)$ that, when paired with $p_t(x)$, satisfies the continuity equation.

Drawing from an adapted theorem based on [(Lipman et al., Theorem 1)](https://arxiv.org/abs/2210.02747), we can establish the existence of such a vector field, as demonstrated in [(Tong et al., Theorem 3.1)](https://arxiv.org/abs/2302.00482). Inheritantly influenced by the structure of $p_t$, $u_t$ takes the form of a mixture involving the conditional vector field $u_t(x | z)$. Formally, it will be defined by:

\begin{equation}\label{eq:mvec}
u_t(x) := \mathbb{E}_{q(z)} \frac{u_t(x | z) p_t(x | z)}{p_t(x)}.
\end{equation}

This formulation ensures that $u_t$ aligns with the theoretical requirements to satisfy the continuity equation when associated with $p_t(x)$. Once more, the complexity introduced by the integral renders the above $u_t(x)$ impractical for direct use. Instead, we rely on use its conditional counterpart $u_t(x|z)$.

In the specefic case where the conditional probability path $p_t(x|z) = \mathcal{N}(\mu_t(z), \sigma_t^2)$ takes the form of a Gaussian, the conditional vector field $u_t(x|z)$ possesses a unique closed-form expression:

\begin{equation}
u_t(x|x_0, x_1) = \frac{\sigma_t'}{\sigma_t} (x - \mu_t) + \mu_t',
\end{equation}

As $\mu_t(x_0, x_1) = t x_1 + (1-t) x_0$ and $\sigma_t = \sigma>0$, the conditional vector field $u_t(x∣z)$ simplifies further:

\begin{equation}
u_t(x|x_0, x_1) = x_1 - x_0.
\end{equation}


The computation of the unconditional vector field $u_t(x)$ and probability path $p_t(x)$ are intractable. Nevertheless, we can easily sample from the conditional probability path $p_t(x|z)$, given its Gaussian nature, and compute the conditional vector field $u_t(x|z)$ with the available closed-form expression.  Consequently, instead of regressing against the unconditional vector field, we opt to regress our network $v_\theta$ against the $\textit{conditional vector field}$ $u_t(x|z)$. The Conditional Flow Matching loss is thus defined as:

$$
\mathcal{L}_{\text{CFM}}(\theta) := \mathbb{E}_{t, q(z), p_t(x | z)} \|v_\theta(x_t) - u_t(x | z)\|^2.
$$

Remarkably, the expectation in this loss reflects the definition of $u_t(x)$. Furthermore, this loss is equal to the unconditional Flow Matching loss, up to a constant that is independent of $\theta$ (Theorem 2, [(Lipman et al.)](https://arxiv.org/abs/2210.02747), Theorem 3.2 [(Tong et al.)](https://arxiv.org/abs/2302.00482)).

It's important to note that the constructed marginal probability path $p_t(x)$ and the derived vector field $u_t(x)$ do not encompass all probability paths and vector fields that satisfy the continuity equation as defined in the ODE section.

In [ ]:
# If there are import errors, please uncomment the following lines to install the required packages.
!pip install pot
!pip install torchdyn
!git clone https://github.com/atong01/conditional-flow-matching.git
%cd conditional-flow-matching


In [ ]:
import math
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import ot as pot
import torch
import torchdyn
from torchdyn.core import NeuralODE
from torchdyn.datasets import generate_moons

from torchcfm.conditional_flow_matching import *
from torchcfm.models.models import *
from torchcfm.utils import *

savedir = "models/8gaussian-moons"
os.makedirs(savedir, exist_ok=True)




In [ ]:
def compute_conditional_vector_field(x0, x1):
    """
    Compute the conditional vector field ut(x1|x0) = x1 - x0, see Eq.(15) [1].

    Parameters
    ----------
    x0 : Tensor, shape (bs, *dim)
        represents the source minibatch
    x1 : Tensor, shape (bs, *dim)
        represents the target minibatch

    Returns
    -------
    ut : conditional vector field ut(x1|x0) = x1 - x0

    References
    ----------
    [1] Improving and Generalizing Flow-Based Generative Models with minibatch optimal transport, Preprint, Tong et al.
    """
    return x1 - x0

In [ ]:
import numpy as np
# Standard_normal and uniform distribution on cicle
def sample_standard_normal(n, dim):
    """ Sample n points from a standard normal distribution in 'dim' dimensions. """
    return torch.randn(n, dim)

def sample_from_circle(n, k, probabilities=None):
    """
    Generates n uniformly distributed points on the unit circle and then samples k points
    from these n points according to a given probability distribution (default is uniform).
    Uses NumPy for generating the points and then converts to PyTorch tensor.
    """
    # Generate n uniformly distributed angles using NumPy
    angles = np.linspace(0, 2 * math.pi, n, endpoint=False)
    x = np.cos(angles)
    y = np.sin(angles)
    points = np.stack((x, y), axis=1)  # Shape: (n, 2)

    # Convert points to a PyTorch tensor
    points_tensor = torch.from_numpy(points).float()

    # If no probabilities are provided, assume uniform distribution
    if probabilities is None:
        probabilities = torch.full((n,), 1.0 / n)  # Even distribution

    # Sample k indices from 0 to n-1 based on the given probabilities
    indices = torch.multinomial(probabilities, k, replacement=True)
    sampled_points = points_tensor[indices]

    return sampled_points.reshape(k, 2)


In [ ]:
def sample_from_points(xi, k):
    """
    Samples k points from the given set of points xi.
    """
    n = xi.shape[0]
    probabilities = torch.full((n,), 1.0 / n)  # Even distribution
    indices = torch.multinomial(probabilities, k, replacement=True)
    # Turn xi into torch float tensor
    xi = torch.tensor(xi, dtype=torch.float)
    return xi[indices]


# Generate Pentagram that contains 10 points
def generate_pentagram(size=1):
    """
    Generates a pentagram containing 10 points.

    Parameters
    ----------
    size : float
        The size of the pentagram.
    """
    # Golden ratio
    phi = (1 + np.sqrt(5)) / 2

    # Degrees for the external points of the star
    external_degrees = np.array([72 * i for i in range(5)])
    # Calculate external points in x and y coordinates using numpy for vectorized operations
    external_points = size * np.array([np.cos(np.radians(external_degrees)), np.sin(np.radians(external_degrees))]).T

    # Calculate internal intersection points
    internal_points = np.zeros_like(external_points)
    for i in range(5):
        j = (i + 2) % 5  # index of the non-adjacent vertex to connect
        # Calculating intersection point along the extended line from vertex i to vertex j
        internal_points[i] = (phi * external_points[i] + external_points[j]) / (1 + phi)

    # Collect all points into a single numpy array
    points = np.vstack([external_points, internal_points])

    # Convert points to a PyTorch tensor
    points_tensor = torch.from_numpy(points).float()

    return points_tensor



def uniformed_points_on_circle(k, shift=[0.0, 0.0]):
    """
    Generates k uniformly distributed points on the unit circle.

    Parameters
    ----------
    k : int
        The number of points to generate.

    Returns
    -------
    xi : torch.Tensor
        The generated points on the unit circle.
    """
    angles = np.linspace(0, 2 * math.pi, k, endpoint=False)
    angles = torch.tensor(angles, dtype=torch.float)

    x_shift = np.array(shift)[0]
    y_shift = np.array(shift)[1]

    xi = torch.stack((torch.cos(angles) + x_shift, torch.sin(angles)+y_shift), dim=1)
    return xi




In [ ]:
import torch.nn as nn


class time_scaled_MLP(torch.nn.Module):
    def __init__(self, dim, out_dim=None, w=64):
        super().__init__()
        out_dim = out_dim if out_dim is not None else dim
        self.net = torch.nn.Sequential(
            torch.nn.Linear(dim + 1, w),
            torch.nn.SELU(),
            torch.nn.Linear(w, w),
            torch.nn.SELU(),
            torch.nn.Linear(w, w),
            torch.nn.SELU(),
            torch.nn.Linear(w, out_dim),
        )

    def forward(self, t, x, args=None):
        # Check that t has the correct shape for concatenation
        # print(t.shape)
        # print(t)

        # print(x.shape)
        # print(x)

        # print(t[:, None].shape)
        # print(t[:, None])



        x_t = torch.cat([x, t[:, None]], dim=-1)
        unscaled_output = self.net(x_t)
        # print(unscaled_output)
        # scale the output according to 1-t
        # print(1-t.unsqueeze(-1))

        scaled_output = unscaled_output / (1 - t[:, None])


        # print(scaled_output)
        return scaled_output


class EarlyLockedVectorField(torch.nn.Module):
    def __init__(self, model, locktime):
        super().__init__()
        self.net = model
        self.locktime = locktime

    def forward(self, t, x, args=None):
        # x: Input features, expected shape [batch_size, dim]
        # t: Time variable, expected shape [batch_size]

        # print(t.shape)
        # print(t)

        # print(x.shape)
        # print(x)

        t = t.repeat(x.shape[0])

        # unscaled_output = self.net(t, x)

        # print(t.shape)
        # print(t)

        if self.locktime < 1:


          t_clamp = torch.clamp(t, 0, self.locktime)

          unscaled_output = self.net(t_clamp, x)* (1 - t_clamp[:, None])

          # print(unscaled_output)

          scaled_output = unscaled_output / (1 - t[:, None])
        else:
          unscaled_output = self.net(t, x)
          scaled_output = unscaled_output

        return scaled_output

In [ ]:
import matplotlib.pyplot as plt


def sample_from_points_with_weight(xi, m, k=2, p=2):
    """
    Samples m points from the given set of points xi.
    """
    n = xi.shape[0]
    probabilities = torch.full((n,), 1.0 / n)  # Even distribution
    indices = torch.multinomial(probabilities, m, replacement=True)
    # Turn xi into torch float tensor
    xi = torch.tensor(xi, dtype=torch.float)


    xi_double = xi.double()
    dist, ind = torch.cdist(xi_double, xi_double).topk(k+1, largest=False)
    dist_pow = dist**p
    dist_to_k_nearest_neighbors = torch.sum(dist_pow, dim=1)
    dist_to_k_nearest_neighbors = (dist_to_k_nearest_neighbors).pow(1/p)

    # normalize the distance to k-nearest neighbors
    dist_to_k_nearest_neighbors = dist_to_k_nearest_neighbors / torch.min(dist_to_k_nearest_neighbors)

    dist_to_k_nearest_neighbors = torch.tensor(dist_to_k_nearest_neighbors,dtype=torch.float)

    return xi[indices], dist_to_k_nearest_neighbors[indices]

def get_ut_xt(x0, x1, t, method='ot-constant', a=19.9, b=0.1, c=10.0, weight=None):
    """
    Draw a sample from the probability path with the specified method.
    """
    assert method in ['ot-constant', 'ot-exp', 'vp-constant', 'vp-exp', 'sub-vp-constant', 'sub-vp-exp', 'modified-ot-constant', 'decoupled-ot-constant', 'decoupled-adaptive-ot-constant']

    t = t.reshape(-1, *([1] * (x0.dim() - 1)))
    if method == 'ot-constant':
        at = t
        bt = 1 - at
        at_prime = 1.0
        bt_prime = -at_prime
    elif method == 'ot-exp':
        at = torch.exp(-0.25*a*(1-t)**2 - 0.5 * b * (1-t))
        bt = 1 - at
        at_prime = (0.5*a*(1-t) + 0.5*b)*at
        bt_prime = -at_prime
    elif method == 'vp-constant':
        at = t
        bt = 1 - at**2
        at_prime = 1.0
        bt_prime = -2*at*at_prime
    elif method == 'vp-exp':
        at = torch.exp(-0.25*a*(1-t)**2 - 0.5 * b * (1-t))
        bt = 1 - at**2
        at_prime = (0.5*a*(1-t) + 0.5*b)*at
        bt_prime = -2*at*at_prime
    elif method == 'sub-vp-constant':
        at = t
        bt = torch.sqrt(1 - at**2)
        at_prime = 1.0
        bt_prime = -at*at_prime/bt
    elif method == 'sub-vp-exp':
        at = torch.exp(-0.25*a*(1-t)**2 - 0.5 * b * (1-t))
        bt = torch.sqrt(1 - at**2)
        at_prime = (0.5*a*(1-t) + 0.5*b)*at
        bt_prime = -at*at_prime/bt
    elif method == 'modified-ot-constant':
        print("Modified OT, To be implemented")
    elif method == 'decoupled-ot-constant':
        at = torch.tanh(c * t)
        at_prime = c * (1 - at**2)
        bt = 1 - t
        bt_prime = -1.0
    elif method == 'decoupled-adaptive-ot-constant':
        at = torch.tanh(c * t)
        # print('at_shape', at.shape)
        # print('t_shape', t.shape)
        # print('weight_shape', weight.shape)
        # print('t**weight_shape', t**weight_shape)
        # assert 1 == 0
        at_prime = c * (1 - at**2)
        bt = (1 - t**weight.unsqueeze(-1))
        bt_prime = -weight.unsqueeze(-1) * t**(weight.unsqueeze(-1)-1)
    xt = at * x1 + bt * x0
    ut = at_prime * x1 + bt_prime * x0

    return xt, ut


In [ ]:
def plot_trajectories_with_xi(traj, xi, loc=None, savedir='.'):
    """
    Plot trajectories of some selected samples.

    Parameters:
    traj -- numpy array of trajectories.
    xi -- numpy array of empirical points.
    loc -- tuple of (x, y) coordinates for focusing the plot around a specific location.
    savedir -- directory where the image will be saved.
    """
    n = 2000
    plt.figure(figsize=(15, 6))
    plt.scatter(traj[0, :n, 0], traj[0, :n, 1], s=10, alpha=0.8, c="black", label="Initial positions")
    plt.scatter(traj[:, :n, 0], traj[:, :n, 1], s=0.2, alpha=0.2, c="olive", label="Trajectory Flow")
    plt.scatter(traj[-1, :n, 0], traj[-1, :n, 1], s=4, alpha=1, c="blue", label="Terminal locations")
    # plot additional reference points
    plt.scatter(xi[:, 0], xi[:, 1], s=10, c="red", label="Empirical points")
    plt.legend()

    if loc is not None:
        x_center, y_center = loc
        delta = 1.1
        plt.xlim(x_center - 5*delta, x_center + 5*delta)
        plt.ylim(y_center - delta, y_center + delta)

    plt.xticks([])
    plt.yticks([])

    # Save high-resolution image in the specified directory
    plt.savefig(f"{savedir}/trajectories_overfit.png", dpi=600)
    plt.show()



In [ ]:
def get_vector_field(model, t, method='ot-constant', a=19.9, b=0.1, c=10.0, weight=None):
    """
    Draw a sample from the probability path with the specified method.
    """
    assert method in ['ot-constant', 'ot-exp', 'vp-constant', 'vp-exp', 'sub-vp-constant', 'sub-vp-exp', 'modified-ot-constant', 'decoupled-ot-constant', 'decoupled-adaptive-ot-constant']

    # t = t.reshape(-1, *([1] * (x0.dim() - 1)))
    t = t.repeat(x.shape[0])
    if method == 'ot-constant':
        at = t
        bt = 1 - at
        at_prime = 1.0
        bt_prime = -at_prime
    elif method == 'ot-exp':
        at = torch.exp(-0.25*a*(1-t)**2 - 0.5 * b * (1-t))
        bt = 1 - at
        at_prime = (0.5*a*(1-t) + 0.5*b)*at
        bt_prime = -at_prime
    elif method == 'vp-constant':
        at = t
        bt = 1 - at**2
        at_prime = 1.0
        bt_prime = -2*at*at_prime
    elif method == 'vp-exp':
        at = torch.exp(-0.25*a*(1-t)**2 - 0.5 * b * (1-t))
        bt = 1 - at**2
        at_prime = (0.5*a*(1-t) + 0.5*b)*at
        bt_prime = -2*at*at_prime
    elif method == 'sub-vp-constant':
        at = t
        bt = torch.sqrt(1 - at**2)
        at_prime = 1.0
        bt_prime = -at*at_prime/bt
    elif method == 'sub-vp-exp':
        at = torch.exp(-0.25*a*(1-t)**2 - 0.5 * b * (1-t))
        bt = torch.sqrt(1 - at**2)
        at_prime = (0.5*a*(1-t) + 0.5*b)*at
        bt_prime = -at*at_prime/bt
    elif method == 'modified-ot-constant':
        print("Modified OT, To be implemented")
    elif method == 'decoupled-ot-constant':
        at = torch.tanh(c * t)
        at_prime = c * (1 - at**2)
        bt = 1 - t
        bt_prime = -1.0
    elif method == 'decoupled-adaptive-ot-constant':
        at = torch.tanh(c * t)
        # print('at_shape', at.shape)
        # print('t_shape', t.shape)
        # print('weight_shape', weight.shape)
        # print('t**weight_shape', t**weight_shape)
        # assert 1 == 0
        at_prime = c * (1 - at**2)
        bt = (1 - t**weight.unsqueeze(-1))
        bt_prime = -weight.unsqueeze(-1) * t**(weight.unsqueeze(-1)-1)
    xt = at * x1 + bt * x0
    ut = at_prime * x1 + bt_prime * x0

In [ ]:
%%time
sigma = 0.0
dim = 2
batch_size = 256

# Define the vector field model as a simple MLP
# vf_model = MLP(dim=dim, time_varying=True)


vf_model = time_scaled_MLP(dim=dim)


optimizer = torch.optim.Adam(vf_model.parameters(), weight_decay=5e-4)


# xi = uniformed_points_on_circle(8)

# shift = [0.0, 10.0]


# xi = uniformed_points_on_circle(8, shift)


# set xi to be ellipse with a=2, b=1

# xi = torch.stack([5*torch.cos(torch.linspace(0, 2*math.pi, 50)) + shift[0], torch.sin(torch.linspace(0, 2*math.pi, 50)) + shift[1]], dim=1)



# create  Cassini ovals usign the parametrization r(t)·(cos(t), sin(t)),
# r^2 (t) := a^2 cos(2t) + sqrt(p(−a^4 + b^4 ) + a^4 (cos(2t))^2) , in https://www.math.uni-bonn.de/people/karcher/VMMPlaneCurves/docs/Cassinian_Oval.pdf

a = 1.0
b = 1.05

t = torch.linspace(0, 2 * math.pi, 20)
r = torch.sqrt(a**2 * torch.cos(2*t) + torch.sqrt(b**4 - a**4 + a**4 * torch.cos(2*t)**2))
xi = torch.stack([r * torch.cos(t), r * torch.sin(t)], dim=1)



# plot
plt.figure(figsize=(12, 6))
plt.scatter(xi[:, 0], xi[:, 1], s=10, c="red")
# add x and y axis
plt.axhline(0, color='black', lw=0.5)
plt.axvline(0, color='black', lw=0.5)

plt.show()



method = 'ot-constant'
print(method)

start = time.time()
for k in range(3000000):
    optimizer.zero_grad()

    x0 = sample_standard_normal(batch_size, 2)

    # x1 = sample_from_points(xi, batch_size)

    x1, weight = sample_from_points_with_weight(xi, batch_size)

    t = torch.rand(x0.shape[0]).type_as(x0)

    # If we want to avoid using t values close to 1 for training,
    # we can set t = t/\delta, where \delta is a value larger than 1.

    t = t


    # xt = sample_conditional_pt(x0, x1, t, sigma)
    # ut = compute_conditional_vector_field(x0, x1)



    # xt, ut = get_ut_xt(x0, x1, t, method=method, a=19.9, b=0.1, c=10.0, weight=weight)
    xt, ut = get_ut_xt(x0, x1, t, method='ot-constant', a=19.9, b=0.1, c=10.0, weight=weight)

    # vt = vf_model(torch.cat([xt, t[:, None]], dim=-1))
    vt = vf_model(t, xt)
    loss = torch.mean((vt - ut) ** 2)

    loss.backward()
    optimizer.step()

    if (k + 1) % 5000 == 0:

        vf = EarlyLockedVectorField(vf_model, 1.0)


        end = time.time()
        print(f"{k+1}: loss {loss.item():0.3f} time {(end - start):0.2f}")
        start = end
        # Load the vector field model into a NeuralODE object
        # node = NeuralODE(
        #     torch_wrapper(vf_model), solver="rk4", sensitivity="adjoint", atol=1e-4, rtol=1e-4
        # )
        node = NeuralODE(
            vf, solver="midpoint", sensitivity="adjoint", atol=1e-4, rtol=1e-4
        )

        with torch.no_grad():
            traj = node.trajectory(
                sample_standard_normal(1024, 2),
                t_span=torch.linspace(0, 1, 1000),
            )
            # plot_trajectories(traj.cpu().numpy())
            plot_trajectories_with_xi(traj.cpu().numpy(), xi)

In [ ]:

class rescaled_MLP(torch.nn.Module):
    def __init__(self, model, rescale_time):
        super().__init__()
        self.net = model
        self.rescale_time = rescale_time

    def forward(self, t, x, args=None):
        # x: Input features, expected shape [batch_size, dim]
        # t: Time variable, expected shape [batch_size]

        t = t.repeat(x.shape[0])

        if self.rescale_time < 1.0:


          t_clamp = torch.clamp(t, self.rescale_time, 1)

          unscaled_output = self.net(t_clamp, x)

          # Unscaled_output = unscaled_output* (1 - t_clamp[:, None])

          # print(unscaled_output)

          scaled_output = unscaled_output
          # scaled_output = unscaled_output / (1 - t[:, None])

        else:
          unscaled_output = self.net(t, x)
          scaled_output = unscaled_output
        return scaled_output

vf = rescaled_MLP(vf_model, 0.999)
# Load the vector field model into a NeuralODE object
# node = NeuralODE(
#     torch_wrapper(vf_model), solver="rk4", sensitivity="adjoint", atol=1e-4, rtol=1e-4
# )
node = NeuralODE(
    vf, solver="rk4", sensitivity="adjoint", atol=1e-4, rtol=1e-4
)

with torch.no_grad():
    traj = node.trajectory(
        sample_standard_normal(1024, 2),
        t_span=torch.linspace(0, 1, 1000),
    )
    # plot_trajectories(traj.cpu().numpy())
    plot_trajectories_with_xi(traj.cpu().numpy(), xi, loc=shift)

In [ ]:

# Plot vector near some locations

def plot_local_vector_field(node, loc, density=1, time_steps=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]):
    """
    Plots the local vector field around the given location.

    Parameters
    ----------
    vector_field : NeuralODE
        The vector field model.
    loc : torch.Tensor
        The location around which to plot the vector field.
    density : int
        The density of the streamplot. Default is 1.
        A higher value will result in more streamlines.
    time_steps : list
        The time steps at which to plot the vector field.
    """
    # Determine the number of rows needed
    rows = (len(time_steps) + 4) // 5  # At most 5 per row
    cols = min(len(time_steps), 5)

    loc = torch.tensor(loc).float().reshape(1, 2)

    # Plotting setup
    fig, axes = plt.subplots(rows, cols, figsize=(15 * cols, 6 * rows))

    for i, t in enumerate(time_steps):
        # Evaluate the vector field at the given location

        # make a grid of points around the location
        n_pts = 10
        neighbor_size = 1.1
        # x = torch.linspace(loc[0, 0] - 5*neighbor_size, loc[0, 0] + 5*neighbor_size, 5*n_pts)
        # y = torch.linspace(loc[0, 1] - neighbor_size, loc[0, 1] + neighbor_size, 2*n_pts)

        # use arrange to avoid precition issue

        x = torch.arange(loc[0, 0] - 5*neighbor_size, loc[0, 0] + 5*neighbor_size, 2*neighbor_size/n_pts)
        y = torch.arange(loc[0, 1] - neighbor_size, loc[0, 1] + neighbor_size, neighbor_size/n_pts)

        nx = len(x)
        ny = len(y)
        X, Y = torch.meshgrid(x, y)

        z = torch.cat([X.reshape(-1, 1), Y.reshape(-1, 1)], 1)
        print(z)
        print(z.shape)
        f = node.vf(torch.tensor(t), z).cpu().detach()

        # Locate the current subplot
        if rows > 1:
            ax = axes[i // 5, i % 5]
        else:
            ax = axes[i] if cols > 1 else axes



        # Plot the vector field
        fx, fy = f[:, 0], f[:, 1]
        fx, fy = fx.reshape(nx, ny), fy.reshape(nx, ny)

        strm = ax.streamplot(X.numpy().T, Y.numpy().T, fx.numpy().T, fy.numpy().T, color='black', density=density)
        contour = ax.contourf(X.T, Y.T, torch.sqrt(fx.T**2 + fy.T**2), cmap='plasma')
        fig.colorbar(contour, ax=ax)
        ax.set_title(f'Time = {t}')


node = NeuralODE(
    vf, solver="rk4", sensitivity="adjoint", atol=1e-4, rtol=1e-4
)
with torch.no_grad():
  traj = node.trajectory(
      sample_standard_normal(1024, 2),
      t_span=torch.linspace(0, 1, 100),
  )
  plot_local_vector_field(node, loc=[0.0, 10.0], density = 1, time_steps=[0.9, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 0.999, 0.9999, 0.99999])


In [ ]:

%%time
sigma = 0.0
dim = 2
batch_size = 256

import matplotlib.pyplot as plt
import torch.nn as nn


class time_scaled_MLP(torch.nn.Module):
    def __init__(self, dim, out_dim=None, w=64):
        super().__init__()
        out_dim = out_dim if out_dim is not None else dim
        self.net = torch.nn.Sequential(
            torch.nn.Linear(dim + 1, w),
            torch.nn.SELU(),
            torch.nn.Linear(w, w),
            torch.nn.SELU(),
            torch.nn.Linear(w, w),
            torch.nn.SELU(),
            torch.nn.Linear(w, out_dim),
        )

    def forward(self, t, x, args=None):
        # Check that t has the correct shape for concatenation
        # print(t.shape)
        # print(t)

        # print(x.shape)
        # print(x)

        # print(t[:, None].shape)
        # print(t[:, None])



        x_t = torch.cat([x, t[:, None]], dim=-1)
        unscaled_output = self.net(x_t)
        # print(unscaled_output)
        # scale the output according to 1-t
        # print(1-t.unsqueeze(-1))

        scaled_output = unscaled_output / (1 - t[:, None])


        # print(scaled_output)
        return scaled_output


class EarlyLockedVectorField(torch.nn.Module):
    def __init__(self, model, locktime):
        super().__init__()
        self.net = model
        self.locktime = locktime

    def forward(self, t, x, args=None):
        # x: Input features, expected shape [batch_size, dim]
        # t: Time variable, expected shape [batch_size]

        # print(t.shape)
        # print(t)

        # print(x.shape)
        # print(x)

        t = t.repeat(x.shape[0])

        # unscaled_output = self.net(t, x)

        # print(t.shape)
        # print(t)

        if self.locktime < 1:


          t_clamp = torch.clamp(t, 0, self.locktime)

          unscaled_output = self.net(t_clamp, x)* (1 - t_clamp[:, None])

          # print(unscaled_output)

          scaled_output = unscaled_output / (1 - t[:, None])
        else:
          unscaled_output = self.net(t, x)
          scaled_output = unscaled_output

        return scaled_output

def sample_from_points_with_weight(xi, m, k=2, p=2):
    """
    Samples m points from the given set of points xi.
    """
    n = xi.shape[0]
    probabilities = torch.full((n,), 1.0 / n)  # Even distribution
    indices = torch.multinomial(probabilities, m, replacement=True)
    # Turn xi into torch float tensor
    xi = torch.tensor(xi, dtype=torch.float)


    xi_double = xi.double()
    dist, ind = torch.cdist(xi_double, xi_double).topk(k+1, largest=False)
    dist_pow = dist**p
    dist_to_k_nearest_neighbors = torch.sum(dist_pow, dim=1)
    dist_to_k_nearest_neighbors = (dist_to_k_nearest_neighbors).pow(1/p)

    # normalize the distance to k-nearest neighbors
    dist_to_k_nearest_neighbors = dist_to_k_nearest_neighbors / torch.min(dist_to_k_nearest_neighbors)

    dist_to_k_nearest_neighbors = torch.tensor(dist_to_k_nearest_neighbors,dtype=torch.float)

    return xi[indices], dist_to_k_nearest_neighbors[indices]

def get_ut_xt(x0, x1, t, method='ot-constant', a=19.9, b=0.1, c=10.0, weight=None):
    """
    Draw a sample from the probability path with the specified method.
    """
    assert method in ['ot-constant', 'ot-exp', 'vp-constant', 'vp-exp', 'sub-vp-constant', 'sub-vp-exp', 'modified-ot-constant', 'decoupled-ot-constant', 'decoupled-adaptive-ot-constant']

    t = t.reshape(-1, *([1] * (x0.dim() - 1)))
    if method == 'ot-constant':
        at = t
        bt = 1 - at
        at_prime = 1.0
        bt_prime = -at_prime
    elif method == 'ot-exp':
        at = torch.exp(-0.25*a*(1-t)**2 - 0.5 * b * (1-t))
        bt = 1 - at
        at_prime = (0.5*a*(1-t) + 0.5*b)*at
        bt_prime = -at_prime
    elif method == 'vp-constant':
        at = t
        bt = 1 - at**2
        at_prime = 1.0
        bt_prime = -2*at*at_prime
    elif method == 'vp-exp':
        at = torch.exp(-0.25*a*(1-t)**2 - 0.5 * b * (1-t))
        bt = 1 - at**2
        at_prime = (0.5*a*(1-t) + 0.5*b)*at
        bt_prime = -2*at*at_prime
    elif method == 'sub-vp-constant':
        at = t
        bt = torch.sqrt(1 - at**2)
        at_prime = 1.0
        bt_prime = -at*at_prime/bt
    elif method == 'sub-vp-exp':
        at = torch.exp(-0.25*a*(1-t)**2 - 0.5 * b * (1-t))
        bt = torch.sqrt(1 - at**2)
        at_prime = (0.5*a*(1-t) + 0.5*b)*at
        bt_prime = -at*at_prime/bt
    elif method == 'modified-ot-constant':
        print("Modified OT, To be implemented")
    elif method == 'decoupled-ot-constant':
        at = torch.tanh(c * t)
        at_prime = c * (1 - at**2)
        bt = 1 - t
        bt_prime = -1.0
    elif method == 'decoupled-adaptive-ot-constant':
        at = torch.tanh(c * t)
        # print('at_shape', at.shape)
        # print('t_shape', t.shape)
        # print('weight_shape', weight.shape)
        # print('t**weight_shape', t**weight_shape)
        # assert 1 == 0
        at_prime = c * (1 - at**2)
        bt = (1 - t**weight.unsqueeze(-1))
        bt_prime = -weight.unsqueeze(-1) * t**(weight.unsqueeze(-1)-1)
    xt = at * x1 + bt * x0
    ut = at_prime * x1 + bt_prime * x0

    return xt, ut


def plot_trajectories_with_xi(traj, xi, loc=None, savedir='.'):
    """
    Plot trajectories of some selected samples.

    Parameters:
    traj -- numpy array of trajectories.
    xi -- numpy array of empirical points.
    loc -- tuple of (x, y) coordinates for focusing the plot around a specific location.
    savedir -- directory where the image will be saved.
    """
    n = 2000
    plt.figure(figsize=(15, 6))
    plt.scatter(traj[0, :n, 0], traj[0, :n, 1], s=10, alpha=0.8, c="black", label="Initial positions")
    plt.scatter(traj[:, :n, 0], traj[:, :n, 1], s=0.2, alpha=0.2, c="olive", label="Trajectory Flow")
    plt.scatter(traj[-1, :n, 0], traj[-1, :n, 1], s=4, alpha=1, c="blue", label="Terminal locations")
    # plot additional reference points
    plt.scatter(xi[:, 0], xi[:, 1], s=10, c="red", label="Empirical points")
    plt.legend()

    if loc is not None:
        x_center, y_center = loc
        delta = 1.1
        plt.xlim(x_center - 5*delta, x_center + 5*delta)
        plt.ylim(y_center - delta, y_center + delta)

    plt.xticks([])
    plt.yticks([])

    # Save high-resolution image in the specified directory
    plt.savefig(f"{savedir}/trajectories_overfit.png", dpi=600)
    plt.show()


# Define the vector field model as a simple MLP
# vf_model = MLP(dim=dim, time_varying=True)


vf_model = time_scaled_MLP(dim=dim)


optimizer = torch.optim.Adam(vf_model.parameters(), weight_decay=5e-4)


# xi = uniformed_points_on_circle(8)

shift = [0.0, 10.0]


# xi = uniformed_points_on_circle(8, shift)


# set xi to be ellipse with a=2, b=1

xi = torch.stack([5*torch.cos(torch.linspace(0, 2*math.pi, 50)) + shift[0], torch.sin(torch.linspace(0, 2*math.pi, 50)) + shift[1]], dim=1)

start = time.time()
for k in range(3000000):
    optimizer.zero_grad()

    x0 = sample_standard_normal(batch_size, 2)

    # x1 = sample_from_points(xi, batch_size)

    x1, weight = sample_from_points_with_weight(xi, batch_size)

    t = torch.rand(x0.shape[0]).type_as(x0)

    # If we want to avoid using t values close to 1 for training,
    # we can set t = t/\delta, where \delta is a value larger than 1.

    t = t/1.001


    # xt = sample_conditional_pt(x0, x1, t, sigma)
    # ut = compute_conditional_vector_field(x0, x1)

    xt, ut = get_ut_xt(x0, x1, t, method='decoupled-ot-constant', a=19.9, b=0.1, c=10.0, weight=weight)
    # xt, ut = get_ut_xt(x0, x1, t, method='ot-constant', a=19.9, b=0.1, c=10.0, weight=weight)

    # vt = vf_model(torch.cat([xt, t[:, None]], dim=-1))
    vt = vf_model(t, xt)
    loss = torch.mean((vt - ut) ** 2)

    loss.backward()
    optimizer.step()

    if (k + 1) % 50000 == 0:

        vf = EarlyLockedVectorField(vf_model, 1.0)


        end = time.time()
        print(f"{k+1}: loss {loss.item():0.3f} time {(end - start):0.2f}")
        start = end
        # Load the vector field model into a NeuralODE object
        # node = NeuralODE(
        #     torch_wrapper(vf_model), solver="rk4", sensitivity="adjoint", atol=1e-4, rtol=1e-4
        # )
        node = NeuralODE(
            vf, solver="rk4", sensitivity="adjoint", atol=1e-4, rtol=1e-4
        )

        with torch.no_grad():
            traj = node.trajectory(
                sample_standard_normal(1024, 2),
                t_span=torch.linspace(0, 1, 1000),
            )
            # plot_trajectories(traj.cpu().numpy())
            plot_trajectories_with_xi(traj.cpu().numpy(), xi, loc=shift)

In [ ]:


# Define the vector field model as a simple MLP
# vf_model = MLP(dim=dim, time_varying=True)


vf_model = time_scaled_MLP(dim=dim)


optimizer = torch.optim.Adam(vf_model.parameters(), weight_decay=5e-4)


# xi = uniformed_points_on_circle(8)

shift = [0.0, 10.0]


# xi = uniformed_points_on_circle(8, shift)


# set xi to be ellipse with a=2, b=1

xi = torch.stack([5*torch.cos(torch.linspace(0, 2*math.pi, 50)) + shift[0], torch.sin(torch.linspace(0, 2*math.pi, 50)) + shift[1]], dim=1)

start = time.time()
for k in range(3000000):
    optimizer.zero_grad()

    x0 = sample_standard_normal(batch_size, 2)

    # x1 = sample_from_points(xi, batch_size)

    x1, weight = sample_from_points_with_weight(xi, batch_size)

    t = torch.rand(x0.shape[0]).type_as(x0)

    # If we want to avoid using t values close to 1 for training,
    # we can set t = t/\delta, where \delta is a value larger than 1.

    t = t/1.001


    # xt = sample_conditional_pt(x0, x1, t, sigma)
    # ut = compute_conditional_vector_field(x0, x1)

    method = 'ot-constant'
    print(method)


    xt, ut = get_ut_xt(x0, x1, t, method=method, a=19.9, b=0.1, c=10.0, weight=weight)
    # xt, ut = get_ut_xt(x0, x1, t, method='ot-constant', a=19.9, b=0.1, c=10.0, weight=weight)

    # vt = vf_model(torch.cat([xt, t[:, None]], dim=-1))
    vt = vf_model(t, xt)
    loss = torch.mean((vt - ut) ** 2)


    loss.backward()
    optimizer.step()

    if (k + 1) % 50000 == 0:

        vf = EarlyLockedVectorField(vf_model, 1.0)


        end = time.time()
        print(f"{k+1}: loss {loss.item():0.3f} time {(end - start):0.2f}")
        start = end
        # Load the vector field model into a NeuralODE object
        # node = NeuralODE(
        #     torch_wrapper(vf_model), solver="rk4", sensitivity="adjoint", atol=1e-4, rtol=1e-4
        # )
        node = NeuralODE(
            vf, solver="rk4", sensitivity="adjoint", atol=1e-4, rtol=1e-4
        )

        with torch.no_grad():
            traj = node.trajectory(
                sample_standard_normal(1024, 2),
                t_span=torch.linspace(0, 1, 1000),
            )
            # plot_trajectories(traj.cpu().numpy())
            plot_trajectories_with_xi(traj.cpu().numpy(), xi, loc=shift)

In [ ]:

# plot multip ple trajectories and save them

def plot_trajectories_loc(traj_list, clamp_limits, loc=None, xi=None, method=None, dataset=None):
    """Plot trajectories of some selected samples."""
    n = 2000

    assert len(traj_list) == len(clamp_limits)

    # Determine the number of rows needed
    rows = (len(traj_list) + 4) // 5  # At most 5 per row
    cols = min(len(traj_list), 5)

    # fig, axs = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    fig, axs = plt.subplots(rows, cols, figsize=(cols * 8, rows * 4))

    for i, (traj, clamp_limit) in enumerate(zip(traj_list, clamp_limits)):
        if rows == 1:
            ax = axs[i]
        else:
            ax = axs[i // cols, i % cols]


        ax.scatter(xi[:, 0], xi[:, 1], s=10, c="red")
        ax.scatter(traj[0, :n, 0], traj[0, :n, 1], s=10, alpha=0.8, c="black")
        ax.scatter(traj[:, :n, 0], traj[:, :n, 1], s=0.2, alpha=0.05, c="olive")
        ax.scatter(traj[-1, :n, 0], traj[-1, :n, 1], s=4, alpha=1, c="blue")
        # ax.legend(["emperical sample", "Flow", "Terminal locations"])
        ax.set_title(f"lock time: {clamp_limit}")
        if loc is not None:
            x_center, y_center = loc
            delta = 1.1
            # ax.set_xlim(x_center - delta, x_center + delta)
            ax.set_xlim(x_center - 5*delta, x_center + 5*delta)
            ax.set_ylim(y_center - delta, y_center + delta)
        ax.set_xticks([])
        ax.set_yticks([])
        # make the  font size in title larger
        ax.title.set_size(20)
    plt.tight_layout()

    filename = f"{savedir}/{method}_{dataset}.jpg"
    plt.savefig(filename, dpi=200)
    plt.show()



class Emperical_optimal_VF(torch.nn.Module):
    def __init__(self, xi, clamp_limit=1.0, method='ot-constant', a=19.9, b=0.1):
        super().__init__()
        self.xi = torch.nn.Parameter(xi).requires_grad_(False)          # xi: Preset points, shape [N, dim]
        self.clamp_limit = clamp_limit
        self.method = method
        self.a = a
        self.b = b

    def forward(self, t, x, args=None):

        assert self.method in ['ot-constant', 'ot-exp', 'vp-constant', 'vp-exp', 'sub-vp-constant', 'sub-vp-exp', 'modified-ot-constant', 'decoupled-ot-constant', 'decoupled-adaptive-ot-constant']

        # x: Input features, expected shape [batch_size, dim]
        # t: Time variable, expected shape [batch_size]

        # Cap the time variable to [1, 1 - eps] to avoid numerical instability in the softmax
        t_clamp = torch.clamp(t, 0, self.clamp_limit)


        if self.method == 'ot-constant':
            # Calculate the difference term for the weights, handling broadcasting
            diff = self.xi.unsqueeze(0) - x.unsqueeze(1)  # x: [batch_size, 1, dim], xi: [1, N, dim]

            # Compute the norms and apply the softmax directly for stability
            t_xi = t_clamp.unsqueeze(-1) * self.xi.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



            norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

            # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
            weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1)).pow(2), dim=1)
            print('(1 - t_clamp.unsqueeze(-1)).pow(2): ', (1 - t_clamp.unsqueeze(-1)).shape)

            # Final output calculation using weights
            output = ((diff / (1 - t.unsqueeze(-1))) * weights.unsqueeze(-1)).sum(dim=1)

        elif self.method == 'ot-exp':
            a = self.a
            b = self.b
            at = torch.exp(-0.25*a*(1-t)**2 - 0.5*b*(1-t))
            bt = 1.0 - at
            at_prime = (0.5*a*(1-t) + 0.5*b)*at


            diff = at_prime*(self.xi.unsqueeze(0) - x.unsqueeze(1))/(1.0 - at)

            at_clamp = torch.exp(-0.25*a*(1-t_clamp)**2 - 0.5*b*(1-t_clamp))

            norms_squared = ((x.unsqueeze(1) - at_clamp.unsqueeze(-1) * self.xi.unsqueeze(0)) ** 2).sum(dim=-1)

            weights = torch.softmax(-0.5 * norms_squared / (1 - at_clamp.unsqueeze(-1)).pow(2), dim=1)

            output = ((diff ) * weights.unsqueeze(-1)).sum(dim=1)

        elif self.method == 'vp-constant':
            diff = (self.xi.unsqueeze(0) -  t* x.unsqueeze(1))/(1-t**2)


            # Compute the norms and apply the softmax directly for stability
            t_xi = t_clamp.unsqueeze(-1) * self.xi.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



            norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

            # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
            # weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)
            weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)

            # Final output calculation using weights
            # output = ((diff / torch.sqrt(1 - t.unsqueeze(-1).pow(2))) * weights.unsqueeze(-1)).sum(dim=1)
            output = ((diff ) * weights.unsqueeze(-1)).sum(dim=1)
        elif self.method == 'vp-exp':

            a = self.a
            b = self.b
            at = torch.exp(-0.25*a*(1-t)**2 - 0.5*b*(1-t))
            bt = torch.sqrt(1- at**2)
            at_prime = (0.5*a*(1-t) + 0.5*b)*at

            # Using vp-constant as a reference
            # at = t
            # bt = torch.sqrt(1.0- at**2)
            # at_prime = 1.0


            # diff = (-at * at_prime * x + at_prime *xi)/sqrt(1-at**2)



            diff = (-at * at_prime * x.unsqueeze(1) + at_prime*self.xi.unsqueeze(0))/(1.0 - at**2)


            # Compute the norms and apply the softmax directly for stability

            at_clamp = torch.exp(-0.25*a*(1-t_clamp)**2 - 0.5*b*(1-t_clamp))

            norms_squared = ((x.unsqueeze(1) - at_clamp.unsqueeze(-1) * self.xi.unsqueeze(0)) ** 2).sum(dim=-1)  # [batch_size, N]

            weights = torch.softmax(-0.5 * norms_squared / (1 - at_clamp.unsqueeze(-1).pow(2)), dim=1)

            output = ((diff ) * weights.unsqueeze(-1)).sum(dim=1)
        elif self.method == 'sub-vp-constant':

            diff = ((1+ t**2)*self.xi.unsqueeze(0) -  (2*t)* x.unsqueeze(1))/(1-t**2)

            t_xi = t_clamp.unsqueeze(-1) * self.xi.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]

            norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

            weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)).pow(2), dim=1)

            output = ((diff ) * weights.unsqueeze(-1)).sum(dim=1)

        elif self.method == 'sub-vp-exp':

            a = self.a
            b = self.b
            at = torch.exp(-0.25*a*(1-t)**2 - 0.5*b*(1-t))
            bt = torch.sqrt(1- at**2)
            at_prime = (0.5*a*(1-t) + 0.5*b)*at

            diff = ((1+ at**2)*at_prime*self.xi.unsqueeze(0) -  (2*at*at_prime)* x.unsqueeze(1))/(1-at**2)

            at_clamp = torch.exp(-0.25*a*(1-t_clamp)**2 - 0.5*b*(1-t_clamp))

            norms_squared = ((x.unsqueeze(1) - at_clamp.unsqueeze(-1) * self.xi.unsqueeze(0)) ** 2).sum(dim=-1)

            weights = torch.softmax(-0.5 * norms_squared / (1 - at_clamp.unsqueeze(-1).pow(2)).pow(2), dim=1)

            output = ((diff ) * weights.unsqueeze(-1)).sum(dim=1)

        elif self.method == 'modified-ot-constant':
            # Calculate the difference term for the weights, handling broadcasting
            diff = self.xi.unsqueeze(0) - x.unsqueeze(1)  # x: [batch_size, 1, dim], xi: [1, N, dim]

            # Compute the norms and apply the softmax directly for stability

            # create t_mask such that t_mask = 1 if t > clamp_limit, t otherwise
            # t_mask = torch.where(t > self.clamp_limit, torch.ones_like(t), t)

            t_mask = torch.ones_like(t)

            t_xi = t_mask.unsqueeze(-1) * self.xi.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



            norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

            # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
            weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1)).pow(2), dim=1)


            # Final output calculation using weights
            output = ((diff / (1 - t.unsqueeze(-1))) * weights.unsqueeze(-1)).sum(dim=1)

        elif self.method == 'decoupled-ot-constant':

            # a = self.a
            # b = self.b
            # at = torch.exp(-0.25*a*(1-t)**2 - 0.5*b*(1-t))
            # at_prime = (0.5*a*(1-t) + 0.5*b)*at
            # at_clamp = torch.exp(-0.25*a*(1-t_clamp)**2 - 0.5*b*(1-t_clamp))

            # a = self.a
            # b = self.b
            # at = torch.exp(-0.25*a*(1-t)**5 - 0.5*b*(1-t))
            # at_prime = (5*0.25*a*(1-t)**4 + 0.5*b)*at
            # at_clamp = torch.exp(-0.25*a*(1-t_clamp)**5 - 0.5*b*(1-t_clamp))



            # c = 10.0
            # at = 1 - torch.exp(-c/(1-t))
            # at_prime = c*torch.exp(-c/(1-t))/(1-t)**2
            # at_clamp = 1 - torch.exp(-c/(1-t_clamp))



            # d = 2.0
            # # ft = 1/(d*t) - 1/(1 - d*t)
            # # ft_prime = -1/(d*t**2) + d/(1 - d*t)**2
            # # at = 1/(1 + torch.exp(ft))
            # # at_prime = (-torch.exp(ft))*ft_prime/(1 + torch.exp(ft))**2

            # # combine the computation of at into one expression and if t> 1/d set it as 1
            # at = torch.where(t > 1/d, torch.ones_like(t), 1/(1 + torch.exp(1/(d*t) - 1/(1 - d*t))))
            # print('at: ', at)
            # # at_prime = torch.where(t > 1/d, torch.zeros_like(t), -torch.exp(1/(d*t) - 1/(1 - d*t))*(1/(d*t**2) - d/(1 - d*t)**2)/(1 + torch.exp(1/(d*t) - 1/(1 - d*t)))**2)

            # at_prime = torch.where(t > 1/d, torch.zeros_like(t), -torch.expm1(1/(d*torch.clamp(t, min=1e-6, max=1/d - 1e-6)) - 1/(1 - d*torch.clamp(t, min=1e-6, max=1/d - 1e-6))) * (1/(d*torch.clamp(t, min=1e-6, max=1/d - 1e-6)**2) - d/(1 - d*torch.clamp(t, min=1e-6, max=1/d - 1e-6))**2) / (1 + torch.exp(1/(d*torch.clamp(t, min=1e-6, max=1/d - 1e-6)) - 1/(1 - d*torch.clamp(t, min=1e-6, max=1/d - 1e-6)))**2))


            # # if at_prime is nan, set it to 0
            # at_prime = torch.where(torch.isnan(at_prime), torch.zeros_like(at_prime), at_prime)
            # print('at_prime: ', at_prime)


            # # ft_clamp = 1/(d*t_clamp) - 1/(1 - d*t_clamp)
            # # at_clamp = 1/(1 + torch.exp(ft_clamp))
            # # clamp the time variable
            # at_clamp = torch.where(t_clamp > 1/d, torch.ones_like(t_clamp), 1/(1 + torch.exp(1/(d*t_clamp) - 1/(1 - d*t_clamp))))
            # print('at_clamp: ', at_clamp)


            # use scaled tanh
            a = 10.0
            at = torch.tanh(a*t)
            at_prime = a*(1 - at**2)
            at_clamp = torch.tanh(a*t_clamp)



            # at = t
            # at_prime = 1.0

            # Using vp-constant as a reference
            # at = t
            # at_prime = 1.0
            # at_clamp = t_clamp

            bt = 1-t
            bt_prime = -1.0
            bt_clamp = 1-t_clamp.unsqueeze(-1)
            # print('bt_clamp: ', bt_clamp.shape)

            # bt = torch.sqrt(1 - t**2)
            # bt_prime = -t/bt
            # bt_clamp = torch.sqrt(1 - t_clamp**2).unsqueeze(-1)

            # bt = 1 - t**2
            # bt_prime = -2*t
            # bt_clamp = (1 - t_clamp**2).unsqueeze(-1)



            diff = (-(bt_prime*at - bt*at_prime)* self.xi.unsqueeze(0) + bt_prime*x.unsqueeze(1))/bt
            # print('diff: ', diff.shape)
            # print seperator
            # print('-----------------')
            norms_squared = ((x.unsqueeze(1) - at_clamp.unsqueeze(-1) * self.xi.unsqueeze(0)) ** 2).sum(dim=-1)  # [batch_size, N]
            # print('norms_squared: ', norms_squared)

            weights = torch.softmax(-0.5 * norms_squared / (bt_clamp).pow(2), dim=1)

            # print('weights: ', weights)




            output = ((diff ) * weights.unsqueeze(-1)).sum(dim=1)
            # print('output: ', output)

        elif self.method == 'decoupled-adaptive-ot-constant':



            # a = self.a
            # b = self.b
            # at = torch.exp(-0.25*a*(1-t)**2 - 0.5*b*(1-t))
            # at_prime = (0.5*a*(1-t) + 0.5*b)*at
            # at_clamp = torch.exp(-0.25*a*(1-t_clamp)**2 - 0.5*b*(1-t_clamp))

            # a = self.a
            # b = self.b
            # at = torch.exp(-0.25*a*(1-t)**5 - 0.5*b*(1-t))
            # at_prime = (5*0.25*a*(1-t)**4 + 0.5*b)*at
            # at_clamp = torch.exp(-0.25*a*(1-t_clamp)**5 - 0.5*b*(1-t_clamp))



            # c = 10.0
            # at = 1 - torch.exp(-c/(1-t))
            # at_prime = c*torch.exp(-c/(1-t))/(1-t)**2
            # at_clamp = 1 - torch.exp(-c/(1-t_clamp))



            # d = 2.0
            # # ft = 1/(d*t) - 1/(1 - d*t)
            # # ft_prime = -1/(d*t**2) + d/(1 - d*t)**2
            # # at = 1/(1 + torch.exp(ft))
            # # at_prime = (-torch.exp(ft))*ft_prime/(1 + torch.exp(ft))**2

            # # combine the computation of at into one expression and if t> 1/d set it as 1
            # at = torch.where(t > 1/d, torch.ones_like(t), 1/(1 + torch.exp(1/(d*t) - 1/(1 - d*t))))
            # print('at: ', at)
            # # at_prime = torch.where(t > 1/d, torch.zeros_like(t), -torch.exp(1/(d*t) - 1/(1 - d*t))*(1/(d*t**2) - d/(1 - d*t)**2)/(1 + torch.exp(1/(d*t) - 1/(1 - d*t)))**2)

            # at_prime = torch.where(t > 1/d, torch.zeros_like(t), -torch.expm1(1/(d*torch.clamp(t, min=1e-6, max=1/d - 1e-6)) - 1/(1 - d*torch.clamp(t, min=1e-6, max=1/d - 1e-6))) * (1/(d*torch.clamp(t, min=1e-6, max=1/d - 1e-6)**2) - d/(1 - d*torch.clamp(t, min=1e-6, max=1/d - 1e-6))**2) / (1 + torch.exp(1/(d*torch.clamp(t, min=1e-6, max=1/d - 1e-6)) - 1/(1 - d*torch.clamp(t, min=1e-6, max=1/d - 1e-6)))**2))


            # # if at_prime is nan, set it to 0
            # at_prime = torch.where(torch.isnan(at_prime), torch.zeros_like(at_prime), at_prime)
            # print('at_prime: ', at_prime)


            # # ft_clamp = 1/(d*t_clamp) - 1/(1 - d*t_clamp)
            # # at_clamp = 1/(1 + torch.exp(ft_clamp))
            # # clamp the time variable
            # at_clamp = torch.where(t_clamp > 1/d, torch.ones_like(t_clamp), 1/(1 + torch.exp(1/(d*t_clamp) - 1/(1 - d*t_clamp))))
            # print('at_clamp: ', at_clamp)


            # use scaled tanh
            a = 10.0
            at = torch.tanh(a*t)
            at_prime = a*(1 - at**2)
            at_clamp = torch.tanh(a*t_clamp)



            # at = t
            # at_prime = 1.0

            # Using vp-constant as a reference
            # at = t
            # at_prime = 1.0
            # at_clamp = t_clamp

            # bt = 1-t
            # bt_prime = -1.0
            # bt_clamp = 1-t_clamp.unsqueeze(-1)

            # compute the distance to k-neaest neighbors for each point in xi
            k = 3

            xi_double = xi.double()
            dist, ind = torch.cdist(xi_double, xi_double).topk(k+1, largest=False)


            # dist_to_k_nearest_neighbors = dist[:, -1]

            # change to sum of dist for the dist_to_k_nearest_neighbors
            # dist_to_k_nearest_neighbors = dist.sum(dim=1)

            dist_pow = dist**2
            dist_to_k_nearest_neighbors = torch.sum(dist_pow, dim=1)
            dist_to_k_nearest_neighbors = (dist_to_k_nearest_neighbors).pow(1/2)

            # normalize the distance to k-nearest neighbors
            dist_to_k_nearest_neighbors = dist_to_k_nearest_neighbors / torch.min(dist_to_k_nearest_neighbors)


            # print('dist_to_k_nearest_neighbors: ', dist_to_k_nearest_neighbors)
            # assert 1==0


            # dist_to_k_nearest_neighbors = torch.ones_like(dist_to_k_nearest_neighbors)


            # use the distance to k-nearest neighbors as the scaling factor on the power of the exponential
            bt = (1 - t**(dist_to_k_nearest_neighbors)).unsqueeze(-1)
            # print('dist_to_k_nearest_neighbors: ', dist_to_k_nearest_neighbors)
            # print('bt: ', bt.shape)
            bt_prime = -dist_to_k_nearest_neighbors*t**(dist_to_k_nearest_neighbors - 1)
            # print('bt_prime: ', bt_prime)
            bt_clamp = (1 - t_clamp**(dist_to_k_nearest_neighbors))
            # print('bt_clamp: ', bt_clamp.shape)

            # bt_clamp_alt = (1 - t_clamp).unsqueeze(-1)

            # # check the shape of bt_clamp and bt_clamp_alt
            # print('bt_clamp: ', bt_clamp.shape)
            # print('bt_clamp_alt: ', bt_clamp_alt.shape)

            # assert 1==0

            # print('bt_clamp: ', bt_clamp)

            # bt = torch.sqrt(1 - t**2)
            # bt_prime = -t/bt
            # bt_clamp = torch.sqrt(1 - t_clamp**2).unsqueeze(-1)

            # bt = 1 - t**2
            # bt_prime = -2*t
            # bt_clamp = (1 - t_clamp**2).unsqueeze(-1)



            # print('bt_prime*at: shape: ', (bt_prime*at).shape)
            # print('bt*at_prime: shape: ', (bt*at_prime).shape)
            # print('self.xi.unsqueeze(0): shape: ', self.xi.unsqueeze(0).shape)
            # print('x.unsqueeze(1): shape: ', x.unsqueeze(1).shape)
            # print('x.unsqueeze(1): ', x.unsqueeze(1))


            # Ensure bt_prime*at and bt*at_prime have compatible shapes with self.xi.unsqueeze(0)
            # bt_prime_at = (bt_prime * at).view(1, 50, 1)  # Reshape to [1, 50, 1] for broadcasting
            # bt_at_prime = (bt * at_prime).view(1, 50, 1) # Reshape to [1, 50, 1] for broadcasting

            # Perform the operations
            # diff = -(bt_prime_at * self.xi.unsqueeze(0) - bt_at_prime * self.xi.unsqueeze(0) + bt_prime * x.unsqueeze(1)) / bt.view(1, 50, 1)
            bt_prime_at = (bt_prime * at).unsqueeze(0).unsqueeze(2)  # Shape [1, 50, 1]
            bt_at_prime = (bt * at_prime).unsqueeze(0)  # Corrected shape: at_prime needs to be [50]

            # diff = -(bt_prime.unsqueeze(-1).unsqueeze(-1) * self.xi.unsqueeze(0) - bt.unsqueeze(-1) * at_prime.unsqueeze(-1).unsqueeze(-1) * self.xi.unsqueeze(0) + bt_prime.unsqueeze(-1) * x.unsqueeze(1).unsqueeze(-1)) / bt.unsqueeze(-1)
            # diff = -(bt_prime_at * self.xi.unsqueeze(0) - bt_at_prime * self.xi.unsqueeze(0) + bt_prime.unsqueeze(0).unsqueeze(2) * x.unsqueeze(1)) / bt.unsqueeze(0).unsqueeze(2)


            # diff = bt_prime_at * self.xi.unsqueeze(0) - bt_at_prime * self.xi.unsqueeze(0)
            # diff = -(bt_prime_at * self.xi.unsqueeze(0) - bt_at_prime * self.xi.unsqueeze(0) - bt_prime.unsqueeze(0).unsqueeze(2) * x.unsqueeze(1))
            diff = -(bt_prime_at * self.xi.unsqueeze(0) - bt_at_prime * self.xi.unsqueeze(0) - bt_prime.unsqueeze(0).unsqueeze(2) * x.unsqueeze(1))/bt


            # fix the bug in diff to be of the shape [batch_size, N, dim]



            # print('bt_clamp: ', bt_clamp.shape)

            # bt = torch.sqrt(1 - t**2)
            # bt_prime = -t/bt
            # bt_clamp = torch.sqrt(1 - t_clamp**2).unsqueeze(-1)

            # bt = 1 - t**2
            # bt_prime = -2*t
            # bt_clamp = (1 - t_clamp**2).unsqueeze(-1)


            # bt_alt = 1-t
            # bt_prime_alt = -1.0
            # bt_clamp_alt = 1-t_clamp.unsqueeze(-1)
            # print('1-t_clamp: ', (1-t_clamp).shape)

            # diff_alt = (bt_prime_alt*at - bt_alt*at_prime)* self.xi.unsqueeze(0)
            # diff_alt = (-(bt_prime_alt*at - bt_alt*at_prime)* self.xi.unsqueeze(0) + bt_prime_alt*x.unsqueeze(1))
            # diff_alt = (-(bt_prime_alt*at - bt_alt*at_prime)* self.xi.unsqueeze(0) + bt_prime_alt*x.unsqueeze(1))/bt_alt

            # compare = torch.allclose(diff, diff_alt, rtol=1e-5, atol=1e-5)




            # print('diff: ', diff)
            norms_squared = ((x.unsqueeze(1) - at_clamp.unsqueeze(-1) * self.xi.unsqueeze(0)) ** 2).sum(dim=-1)  # [batch_size, N]
            # print('norms_squared: ', norms_squared)

            # print('norms_squared: ', norms_squared.shape)


            weights = torch.softmax(-0.5 * norms_squared / (bt_clamp).pow(2), dim=1)
            # print('weights: ', weights)




            output = ((diff ) * weights.unsqueeze(-1)).sum(dim=1)
            # print('output: ', output)

        return output




list_clamp_limit = [0.1, 0.2, 0.3, 0.4, 0.5, 0.52, 0.54, 0.56, 0.58, 0.6, 0.62, 0.64, 0.66, 0.68, 0.7, 0.72, 0.74, 0.76, 0.78, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]


# list_clamp_limit = [0.9, 1.0]
# more_refined_clamp_limit from 0.7 to 1.0
# list_clamp_limit = [0.92, 0.94, 0.96, 0.98, 1.0]
# list_clamp_limit = [0.72, 0.74, 0.76, 0.78, 0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 1.0]
# list_clamp_limit = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
# list from 0.8 to 1.0 with step 0.01
# list_clamp_limit = [0.81, 0.82, 0.83, 0.84, 0.85, 0.86, 0.87, 0.88, 0.89, 0.90, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 1.0]


shift = [300.0, 300.0]


# xi = uniformed_points_on_circle(8, shift)


# set xi to be ellipse with a=2, b=1

xi = torch.stack([5*torch.cos(torch.linspace(0, 2*math.pi, 20)) + shift[0], torch.sin(torch.linspace(0, 2*math.pi, 20)) + shift[1]], dim=1)

# make xi to be n point on the x axis centered at 0

# xi = torch.stack([torch.linspace(-5, 5, 10), torch.zeros(10)], dim=1)


# compute the distance to k-neaest neighbors for each point in xi
# k = 2
# dist, ind = torch.cdist(xi, xi).topk(k+1, largest=False)
# dist_to_k_nearest_neighbors = dist[:, -1]

# # normalize the distance to k-nearest neighbors by minimum distance
# dist_to_k_nearest_neighbors = dist_to_k_nearest_neighbors / torch.min(dist_to_k_nearest_neighbors)
# print('dist_to_k_nearest_neighbors: ', dist_to_k_nearest_neighbors)

# plot the dist_to_k_nearest_neighbors
# plt.plot(dist_to_k_nearest_neighbors)
# plt.show()


# plot xi
# plt.scatter(xi[:, 0], xi[:, 1])
# # add lable to each point
# for i, txt in enumerate(dist_to_k_nearest_neighbors):
#     plt.annotate(f"{txt:.2f}", (xi[i, 0], xi[i, 1]))
# plt.show()


method_list = ['decoupled-adaptive-ot-constant']
# method_list = ['decoupled-adaptive-ot-constant', 'decoupled-ot-constant']
# method_list = ['decoupled-ot-constant']
# method_list = ['ot-constant']

# cluster_results = {}

for method in method_list:
    print('method: ', method)
    traj_list = []
    for clamp_limit in list_clamp_limit:
        det_model = Emperical_optimal_VF(xi, clamp_limit=clamp_limit, method=method)

        node = NeuralODE(
                    det_model, solver="euler", sensitivity="adjoint", atol=1e-4, rtol=1e-4
                )
        with torch.no_grad():
            traj = node.trajectory(
                sample_standard_normal(1024, 2),
                t_span=torch.linspace(0, 1, 10000),
                # t_span=torch.linspace(0, 1, 1000).pow(2),
            )

            # append a copy of the trajectory
            traj_list.append(traj)
        # perform a clustering analysis on the final locations of the trajectories and record the numbers of points in each cluster
        # the clustering is done by finding the closest point in xi to each point in the final location of the trajectory
        # cluster_results[clamp_limit] = []
        # for point in xi:
        #     dist = torch.norm(traj[-1] - point, dim=1)
        #     cluster_results[clamp_limit].append(torch.sum(dist < 0.1).item())


    dataset = 'cicle-shifted-3-3-mod'
    plot_trajectories_loc(traj_list, list_clamp_limit, xi=xi, method=method, dataset = dataset, loc=shift)






In [ ]:
# for each end point in the trajectory, find the closest point in xi
# and then count the number of points in each cluster

def nearest_cluster(point, xi):
    min_dist = torch.norm(point - xi[0])
    min_index = 0
    for i in range(1, len(xi)):
        dist = torch.norm(point - xi[i])
        if dist < min_dist:
            min_dist = dist
            min_index = i
    return min_index

# for each end point in the trajectory, find the closest point in xi

end_points = traj[-1]

# print the length of end_points
print('length of end_points: ', len(end_points))

print('end_points: ', end_points)

cluster_results = {}

print('xi: ', xi)
for i in range(len(xi)):
    cluster_results[i] = 0

for point in end_points:
    cluster_results[nearest_cluster(point, xi)] += 1

print(cluster_results)

# plot xi and mark the cluster analysis results on the corresponding points as text
plt.scatter(xi[:, 0], xi[:, 1], s=10, c="red")
for i in range(len(xi)):
    plt.text(xi[i, 0], xi[i, 1], str(cluster_results[i]), fontsize=12)
plt.savefig(f"{savedir}/cluster_analysis.jpg", dpi=200)

In [ ]:
xi = uniformed_points_on_circle(8, shift=[3.0, 3.0])


# set xi to be ellipse with a=2, b=1

# xi = torch.stack([2*torch.cos(torch.linspace(0, 2*math.pi, 20)), torch.sin(torch.linspace(0, 2*math.pi, 20))], dim=1)

method_list = ['modified-ot-constant']

for method in method_list:
    traj_list = []
    for clamp_limit in list_clamp_limit:
        det_model = Emperical_optimal_VF(xi, clamp_limit=clamp_limit, method=method)

        node = NeuralODE(
                    det_model, solver="euler", sensitivity="adjoint", atol=1e-7, rtol=1e-7
                )
        with torch.no_grad():
            traj = node.trajectory(
                sample_standard_normal(2048, 2),
                t_span=torch.linspace(0, 1, 500),
            )

            # append a copy of the trajectory
            traj_list.append(traj)

    dataset = 'cicle-shifted-3-3'
    plot_trajectories_loc(traj_list, list_clamp_limit, xi=xi, method=method, dataset = dataset, loc=[3.0, 3.0])


In [ ]:
for clamp_limit in list_clamp_limit:
    det_model = Emperical_optimal_VF(xi, clamp_limit=clamp_limit, method='vp-constant')

    node = NeuralODE(
                det_model, solver="euler", sensitivity="adjoint", atol=1e-7, rtol=1e-7
            )
    with torch.no_grad():
        traj = node.trajectory(
            sample_standard_normal(1024, 2),
            t_span=torch.linspace(0, 1, 1000),
        )
        # add label for each plot
        print(f"clamp_limit: {clamp_limit}")
        # print(f"stop_time: {stop_time}")
        plot_trajectories_loc(traj.cpu().numpy())

In [ ]:
# list_clamp_limit = [0.9, 1.0]
# more_refined_clamp_limit from 0.7 to 1.0
# list_clamp_limit = [0.92, 0.94, 0.96, 0.98, 1.0]

xi = uniformed_points_on_circle(8, shift=[20.0, 20.0])
list_clamp_limit = [0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 1.0]
# list_clamp_limit = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
# list_clamp_limit = [ 0.99, 0.995, 1.0]
for clamp_limit in list_clamp_limit:
    det_model = Emperical_optimal_VF(xi, clamp_limit=clamp_limit, method='ot')

    node = NeuralODE(
                det_model, solver="euler", sensitivity="adjoint", atol=1e-7, rtol=1e-7
            )
    with torch.no_grad():
        traj = node.trajectory(
            sample_standard_normal(1024, 2),
            t_span=torch.linspace(0, 1, 1000),
        )
        # add label for each plot
        print(f"clamp_limit: {clamp_limit}")
        # print(f"stop_time: {stop_time}")
        plot_trajectories_loc(traj.cpu().numpy())

In [ ]:

xi = uniformed_points_on_circle(8, shift=[3.0, 3.0])
def plot_trajectories_loc(traj, loc=None):
    """Plot trajectories of some selected samples."""
    n = 2000
    # If loc is not none then we center the figure around loc


    plt.figure(figsize=(6, 6))
    plt.scatter(traj[0, :n, 0], traj[0, :n, 1], s=10, alpha=0.8, c="black")
    plt.scatter(traj[:, :n, 0], traj[:, :n, 1], s=0.2, alpha=0.2, c="olive")
    plt.scatter(traj[-1, :n, 0], traj[-1, :n, 1], s=4, alpha=1, c="blue")
    plt.legend(["Prior sample z(S)", "Flow", "z(0)"])

        # Set plot limits if location is provided
    if loc is not None:
        x_center, y_center = loc
        delta = 1.1  # Define how much area around the center you want to display
        plt.xlim(x_center - delta, x_center + delta)
        plt.ylim(y_center - delta, y_center + delta)

    plt.xticks([])
    plt.yticks([])
    plt.show()

class DeterministicModule(torch.nn.Module):
    def __init__(self, xi, clamp_limit=1.0, smoothing_factor=0.0):
        super().__init__()
        self.xi = torch.nn.Parameter(xi).requires_grad_(False)          # xi: Preset points, shape [N, dim]
        self.clamp_limit = clamp_limit
        self.smoothing_factor = smoothing_factor

    def forward(self, t, x, args=None):
        # x: Input features, expected shape [batch_size, dim]
        # t: Time variable, expected shape [batch_size]

        # Cap the time variable to [1, 1 - eps] to avoid numerical instability in the softmax
        t_clamp = torch.clamp(t, 0, self.clamp_limit)

        # If smoothing factor is non-zero, adding a mean zero Gaussian noise with std = smoothing_factor to xi
        if self.smoothing_factor > 0:
            xi_smooth = self.xi + torch.randn_like(self.xi) * self.smoothing_factor
        else:
            xi_smooth = self.xi

        # Calculate the difference term for the weights, handling broadcasting
        diff = xi_smooth.unsqueeze(0) - x.unsqueeze(1)  # x: [batch_size, 1, dim], xi: [1, N, dim]

        # Compute the norms and apply the softmax directly for stability
        t_xi = t_clamp.unsqueeze(-1) * xi_smooth.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



        norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

        # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
        weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)

        # Final output calculation using weights
        output = ((diff / (1 - t.unsqueeze(-1))) * weights.unsqueeze(-1)).sum(dim=1)

        return output





# list_clamp_limit = [0.9, 1.0]
# more_refined_clamp_limit from 0.7 to 1.0
list_clamp_limit = [0.72, 0.74, 0.76, 0.78, 0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 1.0]
# list_stop_time = [0.72, 0.74, 0.76, 0.78, 0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 1.0]
# list_clamp_limit = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
# list_clamp_limit = [ 0.99, 0.995, 1.0]
for clamp_limit in list_clamp_limit:
    det_model = DeterministicModule(xi, clamp_limit=clamp_limit, smoothing_factor=0.0)

    node = NeuralODE(
                det_model, solver="rk4", sensitivity="adjoint", atol=1e-7, rtol=1e-7
            )
    with torch.no_grad():
        traj = node.trajectory(
            sample_standard_normal(10, 2),
            t_span=torch.linspace(0, 1, 1000),
        )
        # add label for each plot
        print(f"clamp_limit: {clamp_limit}")
        # print(f"stop_time: {stop_time}")
        plot_trajectories_loc(traj.cpu().numpy())





In [ ]:

# xi = uniformed_points_on_circle(8, shift=[0.0, 0.0])


# sample 20 points from a vertical line at x = 3

xi = torch.tensor([[x, 0.0] for x in np.linspace(17.5, 18.5, 2)], dtype=torch.float)

# xi = torch.tensor([[x, 0.0] for x in np.linspace(18.5, 19.5, 2)], dtype=torch.float)

# create a list of rotated xi use float tensor
xi_list = [xi]
# for i in range(1, 8):
#     rotation_matrix = torch.tensor([[np.cos(i * np.pi / 4), -np.sin(i * np.pi / 4)], [np.sin(i * np.pi / 4), np.cos(i * np.pi / 4)]], dtype=torch.float)
#     rotated_xi = torch.matmul(xi, rotation_matrix)
#     xi_list.append(rotated_xi)





def plot_trajectories_with_label(traj):
    """Plot trajectories of some selected samples with labels."""
    n = min(traj.shape[1], 2000)  # Safeguard to plot up to 2000 points or the maximum available

    plt.figure(figsize=(6, 6))
    # Plot the first sample trajectory
    plt.scatter(traj[0, :n, 0], traj[0, :n, 1], s=10, alpha=0.8, c="black")
    for i in range(n):
        plt.annotate(f'{i}', (float(traj[0, i, 0]), float(traj[0, i, 1])))

    # Plot the flow
    plt.scatter(traj[:, :n, 0], traj[:, :n, 1], s=0.2, alpha=0.2, c="olive")
    for i in range(n):
        plt.annotate(f'{i}', (float(traj[i, i % traj.shape[1], 0]), float(traj[i, i % traj.shape[1], 1])))

    # Plot the last sample trajectory
    plt.scatter(traj[-1, :n, 0], traj[-1, :n, 1], s=4, alpha=1, c="blue")
    for i in range(n):
        plt.annotate(f'{i}', (float(traj[-1, i, 0]), float(traj[-1, i, 1])))

    # also plot xi
    # plt.scatter(xi[:, 0], xi[:, 1], s=10, c="red")

    plt.legend(["Prior sample z(S)", "Flow", "z(0)"])
    plt.xticks([])
    plt.yticks([])

    plt.show()

class VP_ODE_constantDeterministicModule(torch.nn.Module):
    def __init__(self, xi, clamp_limit=1.0, smoothing_factor=0.0):
        super().__init__()
        self.xi = torch.nn.Parameter(xi).requires_grad_(False)          # xi: Preset points, shape [N, dim]
        self.clamp_limit = clamp_limit
        self.smoothing_factor = smoothing_factor

    def forward(self, t, x, args=None):
        # x: Input features, expected shape [batch_size, dim]
        # t: Time variable, expected shape [batch_size]

        # Cap the time variable to [1, 1 - eps] to avoid numerical instability in the softmax
        t_clamp = torch.clamp(t, 0, self.clamp_limit)

        # If smoothing factor is non-zero, adding a mean zero Gaussian noise with std = smoothing_factor to xi
        if self.smoothing_factor > 0:
            xi_smooth = self.xi + torch.randn_like(self.xi) * self.smoothing_factor
        else:
            xi_smooth = self.xi


        # if t <= 0.0:
        # Calculate the difference term for the weights, handling broadcasting
        # diff = xi_smooth.unsqueeze(0) - x.unsqueeze(1)  # x: [batch_size, 1, dim], xi: [1, N, dim]
        # diff = (xi_smooth.unsqueeze(0) - t*x.unsqueeze(1))/(1-t**2)

        if t <= -1.0:
            # Calculate the difference term for the weights, handling broadcasting
            diff = xi_smooth.unsqueeze(0) - x.unsqueeze(1)

            # Compute the norms and apply the softmax directly for stability
            t_xi = t_clamp.unsqueeze(-1) * xi_smooth.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



            norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

            # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
            # weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)
            weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1)).pow(2), dim=1)

            # Final output calculation using weights
            # output = ((diff / torch.sqrt(1 - t.unsqueeze(-1).pow(2))) * weights.unsqueeze(-1)).sum(dim=1)
            output = ((diff /(1-t)) * weights.unsqueeze(-1)).sum(dim=1)
        else:
            diff = (xi_smooth.unsqueeze(0) -  t* x.unsqueeze(1))/(1-t**2)


            # Compute the norms and apply the softmax directly for stability
            t_xi = t_clamp.unsqueeze(-1) * xi_smooth.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



            norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

            # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
            # weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)
            weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)

            # Final output calculation using weights
            # output = ((diff / torch.sqrt(1 - t.unsqueeze(-1).pow(2))) * weights.unsqueeze(-1)).sum(dim=1)
            output = ((diff ) * weights.unsqueeze(-1)).sum(dim=1)





        #     # Calculate the difference term for the weights, handling broadcasting
        #     diff = xi_smooth.unsqueeze(0) - x.unsqueeze(1)  # x: [batch_size, 1, dim], xi: [1, N, dim]
        #     # diff = (xi_smooth.unsqueeze(0) - t*x.unsqueeze(1))/(1-t**2)


        #     # Compute the norms and apply the softmax directly for stability
        #     t_xi = t_clamp.unsqueeze(-1) * xi_smooth.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



        #     norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

        #     # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
        #     # weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)
        #     weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1)).pow(2), dim=1)

        #     # Final output calculation using weights
        #     output = ((diff / (1-t.unsqueeze(-1))) * weights.unsqueeze(-1)).sum(dim=1)
        #     # output = ((diff / (torch.sqrt(1 - t.unsqueeze(-1).pow(2))+ 1e-6)) * weights.unsqueeze(-1)).sum(dim=1)

        # else:
        #     # Calculate the difference term for the weights, handling broadcasting
        #     # diff = xi_smooth.unsqueeze(0) - x.unsqueeze(1)  # x: [batch_size, 1, dim], xi: [1, N, dim]
        #     diff = (xi_smooth.unsqueeze(0) - t*x.unsqueeze(1))/(1-t**2)


        #     # Compute the norms and apply the softmax directly for stability
        #     t_xi = t_clamp.unsqueeze(-1) * xi_smooth.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



        #     norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

        #     # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
        #     # weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)
        #     weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)

        #     # Final output calculation using weights
        #     # output = ((diff / torch.sqrt(1 - t.unsqueeze(-1).pow(2))) * weights.unsqueeze(-1)).sum(dim=1)
        #     output = ((diff / (torch.sqrt(1 - t.unsqueeze(-1).pow(2))+ 1e-6)) * weights.unsqueeze(-1)).sum(dim=1)

        return output





# list_clamp_limit = [0.9, 1.0]
# more_refined_clamp_limit from 0.7 to 1.0
# list_clamp_limit = [0.98, 1.0]
# list_clamp_limit = [0.72, 0.74, 0.76, 0.78, 0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 1.0]
# smaller list_clamp_limit
list_clamp_limit = [0.1, 0.2, 0.22, 0.24, 0.26, 0.28, 0.30, 0.32, 0.34, 0.36, 0.38, 0.40, 0.42, 0.44, 0.46, 0.48, 0.50, 0.52, 0.54, 0.56, 0.58, 0.60, 0.62, 0.64, 0.66, 0.68, 0.70, 0.72, 0.74, 0.76, 0.78, 0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 1.0]
# list_clamp_limit = [0.9]
# list_clamp_limit = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
# list_clamp_limit = [ 0.99, 0.995, 1.0]

for xj in xi_list:

    for clamp_limit in list_clamp_limit:

        det_model = Emperical_optimal_VF(xj, clamp_limit=clamp_limit, method=
            'vp-constant')

        node = NeuralODE(
                    det_model, solver="euler", sensitivity="adjoint", atol=1e-7, rtol=1e-7
                )
        with torch.no_grad():
            traj = node.trajectory(
                sample_standard_normal(1024, 2),
                t_span=torch.linspace(0, 1, 1000),
            )
            # add label for each plot
            print(f"clamp_limit: {clamp_limit}")
            # print(f"stop_time: {stop_time}")
            plot_trajectories_loc(traj.cpu().numpy(), xi=xj, loc=[18.0, 0.0], clamp_limit=clamp_limit)
            # plot_trajectories(traj.cpu().numpy())





In [ ]:

xi = uniformed_points_on_circle(8, shift=[0.0, 0.0])


# def plot_trajectories_with_label(traj):
#     """Plot trajectories of some selected samples with labels."""
#     n = min(traj.shape[1], 2000)  # Safeguard to plot up to 2000 points or the maximum available

#     plt.figure(figsize=(6, 6))
#     # Plot the first sample trajectory
#     plt.scatter(traj[0, :n, 0], traj[0, :n, 1], s=10, alpha=0.8, c="black")
#     for i in range(n):
#         plt.annotate(f'{i}', (float(traj[0, i, 0]), float(traj[0, i, 1])))

#     # Plot the flow
#     plt.scatter(traj[:, :n, 0], traj[:, :n, 1], s=0.2, alpha=0.2, c="olive")
#     for i in range(n):
#         plt.annotate(f'{i}', (float(traj[i, i % traj.shape[1], 0]), float(traj[i, i % traj.shape[1], 1])))

#     # Plot the last sample trajectory
#     plt.scatter(traj[-1, :n, 0], traj[-1, :n, 1], s=4, alpha=1, c="blue")
#     for i in range(n):
#         plt.annotate(f'{i}', (float(traj[-1, i, 0]), float(traj[-1, i, 1])))

#     plt.legend(["Prior sample z(S)", "Flow", "z(0)"])
#     plt.xticks([])
#     plt.yticks([])
#     plt.show()

class VP_ODE_constantDeterministicModule(torch.nn.Module):
    def __init__(self, xi, clamp_limit=1.0, smoothing_factor=0.0):
        super().__init__()
        self.xi = torch.nn.Parameter(xi).requires_grad_(False)          # xi: Preset points, shape [N, dim]
        self.clamp_limit = clamp_limit
        self.smoothing_factor = smoothing_factor

    def forward(self, t, x, args=None):
        # x: Input features, expected shape [batch_size, dim]
        # t: Time variable, expected shape [batch_size]

        # Cap the time variable to [1, 1 - eps] to avoid numerical instability in the softmax
        t_clamp = torch.clamp(t, 0, self.clamp_limit)

        # If smoothing factor is non-zero, adding a mean zero Gaussian noise with std = smoothing_factor to xi
        if self.smoothing_factor > 0:
            xi_smooth = self.xi + torch.randn_like(self.xi) * self.smoothing_factor
        else:
            xi_smooth = self.xi


        # if t <= 0.0:
        # Calculate the difference term for the weights, handling broadcasting
        # diff = xi_smooth.unsqueeze(0) - x.unsqueeze(1)  # x: [batch_size, 1, dim], xi: [1, N, dim]
        # diff = (xi_smooth.unsqueeze(0) - t*x.unsqueeze(1))/(1-t**2)

        if t <= 0.0:
            # Calculate the difference term for the weights, handling broadcasting
            diff = xi_smooth.unsqueeze(0) - x.unsqueeze(1)

            # Compute the norms and apply the softmax directly for stability
            t_xi = t_clamp.unsqueeze(-1) * xi_smooth.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



            norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

            # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
            # weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)
            weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1)).pow(2), dim=1)

            # Final output calculation using weights
            # output = ((diff / torch.sqrt(1 - t.unsqueeze(-1).pow(2))) * weights.unsqueeze(-1)).sum(dim=1)
            output = ((diff /(1-t)) * weights.unsqueeze(-1)).sum(dim=1)
        else:
            diff = (xi_smooth.unsqueeze(0) -  x.unsqueeze(1))/(1-t)


            # Compute the norms and apply the softmax directly for stability
            t_xi = t_clamp.unsqueeze(-1) * xi_smooth.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



            norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

            # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
            # weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)
            weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)

            # Final output calculation using weights
            # output = ((diff / torch.sqrt(1 - t.unsqueeze(-1).pow(2))) * weights.unsqueeze(-1)).sum(dim=1)
            output = ((diff ) * weights.unsqueeze(-1)).sum(dim=1)



        #     # Calculate the difference term for the weights, handling broadcasting
        #     diff = xi_smooth.unsqueeze(0) - x.unsqueeze(1)  # x: [batch_size, 1, dim], xi: [1, N, dim]
        #     # diff = (xi_smooth.unsqueeze(0) - t*x.unsqueeze(1))/(1-t**2)


        #     # Compute the norms and apply the softmax directly for stability
        #     t_xi = t_clamp.unsqueeze(-1) * xi_smooth.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



        #     norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

        #     # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
        #     # weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)
        #     weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1)).pow(2), dim=1)

        #     # Final output calculation using weights
        #     output = ((diff / (1-t.unsqueeze(-1))) * weights.unsqueeze(-1)).sum(dim=1)
        #     # output = ((diff / (torch.sqrt(1 - t.unsqueeze(-1).pow(2))+ 1e-6)) * weights.unsqueeze(-1)).sum(dim=1)

        # else:
        #     # Calculate the difference term for the weights, handling broadcasting
        #     # diff = xi_smooth.unsqueeze(0) - x.unsqueeze(1)  # x: [batch_size, 1, dim], xi: [1, N, dim]
        #     diff = (xi_smooth.unsqueeze(0) - t*x.unsqueeze(1))/(1-t**2)


        #     # Compute the norms and apply the softmax directly for stability
        #     t_xi = t_clamp.unsqueeze(-1) * xi_smooth.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



        #     norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

        #     # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
        #     # weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)
        #     weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)

        #     # Final output calculation using weights
        #     # output = ((diff / torch.sqrt(1 - t.unsqueeze(-1).pow(2))) * weights.unsqueeze(-1)).sum(dim=1)
        #     output = ((diff / (torch.sqrt(1 - t.unsqueeze(-1).pow(2))+ 1e-6)) * weights.unsqueeze(-1)).sum(dim=1)

        return output





# list_clamp_limit = [0.9, 1.0]
# more_refined_clamp_limit from 0.7 to 1.0
list_clamp_limit = [0.82, 0.84, 0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 1.0]
# list_stop_time = [0.72, 0.74, 0.76, 0.78, 0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 1.0]
# list_clamp_limit = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
# list_clamp_limit = [ 0.99, 0.995, 1.0]
for clamp_limit in list_clamp_limit:
    det_model = VP_ODE_constantDeterministicModule(xi, clamp_limit=clamp_limit, smoothing_factor=0.0)

    node = NeuralODE(
                det_model, solver="rk4", sensitivity="adjoint", atol=1e-7, rtol=1e-7
            )
    with torch.no_grad():
        traj = node.trajectory(
            sample_standard_normal(1024, 2),
            t_span=torch.linspace(0, 1, 500),
        )
        # add label for each plot
        print(f"clamp_limit: {clamp_limit}")
        # print(f"stop_time: {stop_time}")
        plot_trajectories_loc(traj.cpu().numpy(), loc=[0.0, 0.0])





In [ ]:
# Helpper function to plot the vector field
def plot_vector_field(node, traj, density=1,time_steps=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]):
    # Evaluate vector field
    n_pts = 100
    x = torch.linspace(traj[:, :, 0].min(), traj[:, :, 0].max(), n_pts)
    y = torch.linspace(traj[:, :, 1].min(), traj[:, :, 1].max(), n_pts)
    X, Y = torch.meshgrid(x, y)
    z = torch.cat([X.reshape(-1, 1), Y.reshape(-1, 1)], 1)

    # Determine the number of rows needed
    rows = (len(time_steps) + 4) // 5  # At most 5 per row
    cols = min(len(time_steps), 5)

    # Plotting setup
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))

    for i, t in enumerate(time_steps):
        f = node.vf(torch.tensor(t), z).cpu().detach()
        fx, fy = f[:, 0], f[:, 1]
        fx, fy = fx.reshape(n_pts, n_pts), fy.reshape(n_pts, n_pts)

        # Locate the current subplot
        if rows > 1:
            ax = axes[i // 5, i % 5]
        else:
            ax = axes[i] if cols > 1 else axes

        # Plot vector field and its intensity
        strm = ax.streamplot(X.numpy().T, Y.numpy().T, fx.numpy().T, fy.numpy().T, color='black',density=density)
        contour = ax.contourf(X.T, Y.T, torch.sqrt(fx.T**2 + fy.T**2), cmap='plasma')
        fig.colorbar(contour, ax=ax)
        ax.set_title(f'Time = {t}')

    plt.tight_layout()
    plt.show()




# Plot vector near some locations

def plot_local_vector_field(node, loc, density=1, time_steps=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]):
    """
    Plots the local vector field around the given location.

    Parameters
    ----------
    vector_field : NeuralODE
        The vector field model.
    loc : torch.Tensor
        The location around which to plot the vector field.
    density : int
        The density of the streamplot. Default is 1.
        A higher value will result in more streamlines.
    time_steps : list
        The time steps at which to plot the vector field.
    """
    # Determine the number of rows needed
    rows = (len(time_steps) + 4) // 5  # At most 5 per row
    cols = min(len(time_steps), 5)

    loc = torch.tensor(loc).float().reshape(1, 2)

    # Plotting setup
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))

    for i, t in enumerate(time_steps):
        # Evaluate the vector field at the given location

        # make a grid of points around the location
        n_pts = 100
        neighbor_size = 0.5
        x = torch.linspace(loc[0, 0] - neighbor_size, loc[0, 0] + neighbor_size, n_pts)
        y = torch.linspace(loc[0, 1] - neighbor_size, loc[0, 1] + neighbor_size, n_pts)
        X, Y = torch.meshgrid(x, y)
        z = torch.cat([X.reshape(-1, 1), Y.reshape(-1, 1)], 1)
        f = node.vf(torch.tensor(t), z).cpu().detach()

        # Locate the current subplot
        if rows > 1:
            ax = axes[i // 5, i % 5]
        else:
            ax = axes[i] if cols > 1 else axes



        # Plot the vector field
        fx, fy = f[:, 0], f[:, 1]
        fx, fy = fx.reshape(n_pts, n_pts), fy.reshape(n_pts, n_pts)
        strm = ax.streamplot(X.numpy().T, Y.numpy().T, fx.numpy().T, fy.numpy().T, color='black', density=density)
        contour = ax.contourf(X.T, Y.T, torch.sqrt(fx.T**2 + fy.T**2), cmap='plasma')
        fig.colorbar(contour, ax=ax)
        ax.set_title(f'Time = {t}')



def plot_vector_field_variation(node, traj, time_step):
    n_pts = 100
    x = torch.linspace(traj[:, :, 0].min(), traj[:, :, 0].max(), n_pts)
    y = torch.linspace(traj[:, :, 1].min(), traj[:, :, 1].max(), n_pts)
    X, Y = torch.meshgrid(x, y)
    z = torch.cat([X.reshape(-1, 1), Y.reshape(-1, 1)], 1)

    # Compute vector field
    z.requires_grad_(True)
    time_step = torch.tensor(time_step).requires_grad_(True)
    f = node.vf(torch.tensor(time_step), z)

    fx, fy = f[:, 0], f[:, 1]

    # Compute gradients
    fx_grad_x, fx_grad_y = torch.autograd.grad(fx, z, grad_outputs=torch.ones_like(fx), create_graph=True)[0].chunk(2, dim=1)
    fy_grad_x, fy_grad_y = torch.autograd.grad(fy, z, grad_outputs=torch.ones_like(fy), create_graph=True)[0].chunk(2, dim=1)


    # Compute Lipschitz constants
    fx_lipschitz = torch.sqrt(fx_grad_x**2 + fx_grad_y**2).reshape(n_pts, n_pts)
    fy_lipschitz = torch.sqrt(fy_grad_x**2 + fy_grad_y**2).reshape(n_pts, n_pts)




    # Plot vector field and Lipschitz constants
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))

    ax = axs[0]
    ax.streamplot(X.detach().numpy().T, Y.detach().numpy().T, fx.detach().reshape(n_pts, n_pts).numpy().T, fy.detach().reshape(n_pts, n_pts).numpy().T, color='black')
    contour = ax.contourf(X.detach().numpy().T, Y.detach().numpy().T, torch.sqrt(fx.reshape(n_pts, n_pts).detach().T**2 + fy.reshape(n_pts, n_pts).detach().T**2), cmap='plasma')
    fig.colorbar(contour, ax=ax)

    ax.set_title(f"Vector Field at Time Step: {time_step:.2f}")

    ax = axs[1]

    ax.imshow(fx_lipschitz.detach().numpy().T, cmap='plasma', extent=[x.min(), x.max(), y.min(), y.max()], vmax=fx_lipschitz.max(), vmin=fx_lipschitz.min())
    fig.colorbar(ax.imshow(fx_lipschitz.detach().numpy().T, cmap='plasma', extent=[x.min(), x.max(), y.min(), y.max()], vmax=fx_lipschitz.max(), vmin=fx_lipschitz.min()), ax=ax)


    ax.set_title("Lipschitz Constant of x Component")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    ax = axs[2]
    ax.imshow(fy_lipschitz.detach().numpy().T, cmap='plasma', extent=[x.min(), x.max(), y.min(), y.max()], vmax=fy_lipschitz.max(), vmin=fy_lipschitz.min())
    fig.colorbar(ax.imshow(fy_lipschitz.detach().numpy().T, cmap='plasma', extent=[x.min(), x.max(), y.min(), y.max()], vmax=fy_lipschitz.max(), vmin=fy_lipschitz.min()), ax=ax)
    ax.set_title("Lipschitz Constant of y Component")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    # ax = axs[3]



    plt.tight_layout()
    plt.show()


# Plot the vector field and its Lipschitz constants at time step 0.5

plot_vector_field_variation(node, traj, 0.95)

In [ ]:
%%time
sigma = 0.1
dim = 2
batch_size = 256
model = MLP(dim=dim, time_varying=True)
optimizer = torch.optim.Adam(model.parameters())

xi = uniformed_points_on_circle(8)

start = time.time()
for k in range(50000):
    optimizer.zero_grad()

    x0 = sample_standard_normal(batch_size, 2)
    x1 = sample_from_points(xi, batch_size)

    t = torch.rand(x0.shape[0]).type_as(x0)
    xt = sample_conditional_pt(x0, x1, t, sigma=0.00)
    ut = compute_conditional_vector_field(x0, x1)

    vt = model(torch.cat([xt, t[:, None]], dim=-1))
    loss = torch.mean((vt - ut) ** 2)

    loss.backward()
    optimizer.step()

    if (k + 1) % 5000 == 0:
        end = time.time()
        print(f"{k+1}: loss {loss.item():0.3f} time {(end - start):0.2f}")
        start = end
        node = NeuralODE(
            torch_wrapper(model), solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4
        )
        with torch.no_grad():
            traj = node.trajectory(
                sample_standard_normal(1024, 2),
                t_span=torch.linspace(0, 1, 100),
            )
            plot_trajectories(traj.cpu().numpy())

            # evaluate vector field
            plot_local_vector_field(node, [0.0, 1.0], density=1,time_steps=[0.97, 0.98, 0.99, 0.999, 0.9999, 0.99999, 0.999999, 1.0])

        # plot the lipschitz constant
        plot_vector_field_variation(node, traj, 0.99)


In [ ]:
xi = uniformed_points_on_circle(8, shift=[3.0, 3.0])

def plot_trajectories_with_xi(traj, xi):
    """Plot trajectories of some selected samples."""
    n = 2000
    plt.figure(figsize=(6, 6))
    plt.scatter(traj[0, :n, 0], traj[0, :n, 1], s=10, alpha=0.8, c="black")
    plt.scatter(traj[:, :n, 0], traj[:, :n, 1], s=0.2, alpha=0.2, c="olive")
    plt.scatter(traj[-1, :n, 0], traj[-1, :n, 1], s=4, alpha=1, c="blue")
    # plot additional reference points
    plt.scatter(xi[:, 0], xi[:, 1], s=10, c="red")
    plt.legend(["Prior sample z(S)", "Flow", "z(0)"])
    plt.xticks([])
    plt.yticks([])
    plt.show()

class DeterministicModule(torch.nn.Module):
    def __init__(self, xi, clamp_limit=1.0, smoothing_factor=0.0):
        super().__init__()
        self.xi = torch.nn.Parameter(xi)  # xi: Preset points, shape [N, dim]
        self.clamp_limit = clamp_limit
        self.smoothing_factor = smoothing_factor

    def forward(self, t, x, args=None):
        # x: Input features, expected shape [batch_size, dim]
        # t: Time variable, expected shape [batch_size]

        # Cap the time variable to [1, 1 - eps] to avoid numerical instability in the softmax
        t_clamp = torch.clamp(t, 0, self.clamp_limit)


        # If smoothing factor is non-zero, adding a mean zero Gaussian noise with std = smoothing_factor to xi
        if self.smoothing_factor > 0:
            xi_smooth = self.xi + torch.randn_like(self.xi) * self.smoothing_factor
        else:
            xi_smooth = self.xi

        # Calculate the difference term for the weights, handling broadcasting
        diff = xi_smooth.unsqueeze(0) - x.unsqueeze(1)  # x: [batch_size, 1, dim], xi: [1, N, dim]

        # Compute the norms and apply the softmax directly for stability
        t_xi = t_clamp.unsqueeze(-1) * xi_smooth.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



        norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

        # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
        weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)

        # Final output calculation using weights
        output = ((diff / (1 - t.unsqueeze(-1))) * weights.unsqueeze(-1)).sum(dim=1)

        return output





# list_clamp_limit = [0.9, 1.0]
# more_refined_clamp_limit from 0.7 to 1.0
# list_clamp_limit = [0.72, 0.74, 0.76, 0.78, 0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 1.0]
# list_stop_time = [0.72, 0.74, 0.76, 0.78, 0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 1.0]
list_clamp_limit = [0.8, 0.85, 0.9, 0.95, 1.0]
# list_clamp_limit = [ 0.99, 0.995, 1.0]
for clamp_limit in list_clamp_limit:
    det_model = DeterministicModule(xi, clamp_limit=clamp_limit, smoothing_factor=0.0)

    node = NeuralODE(
                det_model, solver="rk4", sensitivity="adjoint", atol=1e-7, rtol=1e-7
            )
    with torch.no_grad():
        traj = node.trajectory(
            sample_standard_normal(1024, 2),
            t_span=torch.linspace(0, 1, 1000),
        )
        # add label for each plot
        print(f"clamp_limit: {clamp_limit}")
        # print(f"stop_time: {stop_time}")
        # plot_trajectories(traj.cpu().numpy())
        plot_trajectories_with_xi(traj.cpu().numpy(), xi=xi)


    # evaluate vector field lipshitz constant
    plot_vector_field_variation(node, traj, 0.99)


In [ ]:
xi = generate_pentagram(size=1)

class DeterministicModule(torch.nn.Module):
    def __init__(self, xi, clamp_limit=1.0, smoothing_factor=0.0):
        super().__init__()
        self.xi = torch.nn.Parameter(xi)  # xi: Preset points, shape [N, dim]
        self.clamp_limit = clamp_limit
        self.smoothing_factor = smoothing_factor

    def forward(self, t, x, args=None):
        # x: Input features, expected shape [batch_size, dim]
        # t: Time variable, expected shape [batch_size]

        # Cap the time variable to [1, 1 - eps] to avoid numerical instability in the softmax
        t_clamp = torch.clamp(t, 0, self.clamp_limit)

        # If smoothing factor is non-zero, adding a mean zero Gaussian noise with std = smoothing_factor to xi
        if self.smoothing_factor > 0:
            xi_smooth = self.xi + torch.randn_like(self.xi) * self.smoothing_factor
        else:
            xi_smooth = self.xi

        # Calculate the difference term for the weights, handling broadcasting
        diff = xi_smooth.unsqueeze(0) - x.unsqueeze(1)  # x: [batch_size, 1, dim], xi: [1, N, dim]

        # Compute the norms and apply the softmax directly for stability
        t_xi = t_clamp.unsqueeze(-1) * xi_smooth.unsqueeze(0)  # t: [batch_size, 1], xi: [1, N, dim]



        norms_squared = ((x.unsqueeze(1) - t_xi) ** 2).sum(dim=-1)  # [batch_size, N]

        # Apply softmax to -0.5 * norms_squared / (1 - t)^2 for numerical stability
        weights = torch.softmax(-0.5 * norms_squared / (1 - t_clamp.unsqueeze(-1).pow(2)), dim=1)

        # Final output calculation using weights
        # output = ((diff / (1 - t.unsqueeze(-1))) * weights.unsqueeze(-1)).sum(dim=1)
        output = (diff / (1 - t.unsqueeze(-1)) * weights.unsqueeze(-1)).sum(dim=1)

        return output


# list_clamp_limit = [0.9, 1.0]
# more_refined_clamp_limit from 0.7 to 1.0
list_clamp_limit = [0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 1.0]
# list_stop_time = [0.72, 0.74, 0.76, 0.78, 0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 1.0]
# list_clamp_limit = [0.8, 0.85, 0.9, 0.95, 1.0]
# list_clamp_limit = [ 0.99, 0.995, 1.0]
for clamp_limit in list_clamp_limit:
    det_model = DeterministicModule(xi, clamp_limit=clamp_limit, smoothing_factor=0.0)

    node = NeuralODE(
                det_model, solver="rk4", sensitivity="adjoint", atol=1e-7, rtol=1e-7
            )
    with torch.no_grad():
        traj = node.trajectory(
            sample_standard_normal(1024, 2),
            t_span=torch.linspace(0, 1, 100),
        )
        # add label for each plot
        print(f"clamp_limit: {clamp_limit}")
        # print(f"stop_time: {stop_time}")
        plot_trajectories(traj.cpu().numpy())

        plot_vector_field(node, traj, time_steps=[0.7, 0.8, 0.9, 0.95, 0.96, 0.97, 0.98, 0.99, 0.999, 0.9999, 0.99999, 0.999999, 1.0])

    # evaluate vector field lipshitz constant
    plot_vector_field_variation(node, traj, 0.99)


In [ ]:
%%time
sigma = 0.1
dim = 2
batch_size = 256
model = MLP(dim=dim, time_varying=True)
optimizer = torch.optim.Adam(model.parameters())

xi = generate_pentagram(size=1.5)

start = time.time()
for k in range(50000):
    optimizer.zero_grad()

    x0 = sample_standard_normal(batch_size, 2)
    x1 = sample_from_points(xi, batch_size)

    t = torch.rand(x0.shape[0]).type_as(x0)
    xt = sample_conditional_pt(x0, x1, t, sigma=0.00)
    ut = compute_conditional_vector_field(x0, x1)

    vt = model(torch.cat([xt, t[:, None]], dim=-1))
    loss = torch.mean((vt - ut) ** 2)

    loss.backward()
    optimizer.step()

    if (k + 1) % 5000 == 0:
        end = time.time()
        print(f"{k+1}: loss {loss.item():0.3f} time {(end - start):0.2f}")
        start = end
        node = NeuralODE(
            torch_wrapper(model), solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4
        )
        with torch.no_grad():
            traj = node.trajectory(
                sample_standard_normal(1024, 2),
                t_span=torch.linspace(0, 1, 100),
            )
            plot_trajectories(traj.cpu().numpy())

            # evaluate vector field
            plot_vector_field(node, traj, time_steps=[0.7, 0.8, 0.9, 0.95, 0.96, 0.97, 0.98, 0.99, 0.999, 0.9999, 0.99999, 0.999999, 1.0])

        # evaluate vector field lipshitz constant
        plot_vector_field_variation(node, traj, 0.99)

In [ ]:
%%time
sigma = 0.1
dim = 2
batch_size = 256
model = MLP(dim=dim, time_varying=True)
optimizer = torch.optim.Adam(model.parameters())

xi = generate_pentagram(size=1.5)

start = time.time()
for k in range(50000):
    optimizer.zero_grad()

    x0 = sample_standard_normal(batch_size, 2)
    x1 = sample_from_points(xi, batch_size)

    t = torch.rand(x0.shape[0]).type_as(x0)

    # If we want to avoid using t values close to 1 for training,
    # we can set t = t/\delta, where \delta is a value larger than 1.

    t = t/1.3

    xt = sample_conditional_pt(x0, x1, t, sigma=0.00)
    ut = compute_conditional_vector_field(x0, x1)

    vt = model(torch.cat([xt, t[:, None]], dim=-1))
    loss = torch.mean((vt - ut) ** 2)

    loss.backward()
    optimizer.step()

    if (k + 1) % 5000 == 0:
        end = time.time()
        print(f"{k+1}: loss {loss.item():0.3f} time {(end - start):0.2f}")
        start = end
        node = NeuralODE(
            torch_wrapper(model), solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4
        )
        with torch.no_grad():
            traj = node.trajectory(
                sample_standard_normal(1024, 2),
                t_span=torch.linspace(0, 1, 100),
            )
            plot_trajectories(traj.cpu().numpy())

            # evaluate vector field
            plot_vector_field(node, traj, time_steps=[0.7, 0.8, 0.9, 0.95, 0.96, 0.97, 0.98, 0.99, 0.999, 0.9999, 0.99999, 0.999999, 1.0])

        # evaluate vector field lipshitz constant
        plot_vector_field_variation(node, traj, 0.99)